# CLCT threshold selection (Youden's J), with parallax and detectability corrections

A focused re-implementation of section 5 (method 4c) of `CLCT_binary_verification.ipynb`: sweep a CLCT
threshold, pick `tau*` maximising Youden's J = POD - POFD on calibration data, and report skill on
evaluation data that took no part in the choice. Everything else from that notebook is dropped so that the
threshold itself can be done properly.

The original quoted `tau*` as a single number from a single split against an uncorrected satellite mask.
Four things limit how much that number means, and each is addressed here:

1. **Parallax.** MSG sees Switzerland at ~54 deg viewing zenith, so a cloud top at height `z` is reported
   about `1.38 z` away from the ground point it actually sits over -- 3-4 grid cells for high cloud. The
   mask is displaced back onto true ground positions using the `ctth_alti` retrieval in the same file
   (section 4). Clear pixels are displaced too, by terrain height, which is not negligible in the Alps.
2. **Retrieval quality and cloud class.** These files carry **no** `cma_quality` / `cma_conditions`
   variables -- only the CTTH product has flags. What is available (`ctth_quality`, `ctth_conditions`,
   `ctth_status_flag`, `ctth_method`) qualifies the *height*, not the cloud bit, so it gates the parallax
   correction rather than the mask. Screening of the mask itself goes through the `ct` cloud-type product
   instead. Section 3 states exactly what this can and cannot do.
3. **Scan timing.** SEVIRI scans south to north across a 12m23s acquisition, so the Alps are imaged about
   10 minutes after the nominal slot time stamped on the filename, while ICON `lff` is instantaneous on the
   hour. Pairing hour `T` with slot `T` is therefore a ~10 minute mismatch; the `T-15min` slot is closer.
   Section 5 derives this from the files' own `start_time`/`end_time` and picks the better slot.
4. **Detectability.** MSG cannot see what is optically too thin, and NWCSAF says so itself through the
   `Fractional_clouds` / `High_semitransparent_thin` cloud types and the
   `Too_thin_clouds_no_reliable_method` status bit. ICON's CLCT counts that cloud regardless. Section 6
   adds the complementary model-side view: a CLCT recomputed with optically undetectable layers removed.

`tau*` is then reported not as a point value but with a moving-block bootstrap interval and k-fold
cross-validation over time blocks (sections 11-12), because hourly steps are not independent samples and
the J curve is typically flat near its maximum. Section 10 is the point of the whole notebook: the same
threshold refitted under each correction in turn, so the movement in `tau*` is attributable.

**Data actually on disk** (verified, not assumed): NWCSAF 15-min slots cover 2025-10-04 00:00 ..
2025-10-10 00:00 with exactly two missing (2025-10-06 12:00 and 12:15); experiment 801/802 `lff` covers
2025-10-04 01:00 .. 2025-10-10 01:00. The NWCSAF grid is a regular 341x482 lat/lon box at 0.0388 x 0.0270
deg (~2.97 x 3.00 km) spanning 1.08W..17.58E, 41.91N..51.09N, with latitude **descending**. `cma` is a
clean 0/1 field with no fill value in these exports.

In [ ]:
import os
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import shapely
import earthkit.data as ekd

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

if shapely.__version__ < "2":
    raise RuntimeError(f"needs shapely >= 2, found {shapely.__version__}")

# =============================================================================
# Config
# =============================================================================
EXP = "801"
NWCSAF_DIR = "/scratch/mch/jdelbeke/nwcsaf"
ICON_DIR = f"/store_new/mch/msopr/jdelbeke/ICON_TST/{EXP}/FG25/det"
ICON_GRID_FILE = "/oprusers/osm/opr.inn/data/ICON_INPUT/ICON-CH1-EPS/icon_grid_0001_R19B08_mch.nc"

CACHE_DIR = "/scratch/mch/jdelbeke"
WEIGHTS_CACHE = f"{CACHE_DIR}/regrid_weights_icon_nwcsaf.npz"
HIST_CACHE = f"{CACHE_DIR}/clct_threshold_hists_{EXP}.npz"


def nwcsaf_file(t):
    return f"{NWCSAF_DIR}/MSG_nwcsaf_cosmo1eqc3km_{t:%Y%m%d%H%M}.nc"


def icon_file(t):
    return f"{ICON_DIR}/lff{t:%Y%m%d%H}"


# NWCSAF runs out at 2025-10-10 00:00, so the last usable ICON hour is that one
# (and with scan-time matching it pairs with the 2025-10-09 23:45 slot).
START = datetime(2025, 10, 4, 1)
N_HOURS = 144
TIMES = [START + timedelta(hours=h) for h in range(N_HOURS)]
SPLIT_DATE = datetime(2025, 10, 7)     # daytime from here on = evaluation pool

DOMAIN = None            # None = whole NWCSAF grid, or [lonW, lonE, latS, latN]
MIN_COVERAGE = 0.999     # target cells less covered by ICON than this -> NaN
CHUNK = 200_000

# --- threshold grid ---------------------------------------------------------
# CLCT is 0..100. Binning it lets every J evaluation below (bootstrap,
# cross-validation, every stratum) be a sum over precomputed per-timestep
# histograms rather than a fresh pass over the pixels.
#
# Binning is exact, not approximate: bins are floor(CLCT * BIN_PER_PCT) and the
# threshold is lower-inclusive, so "CLCT >= TAUS[k]" is identically "bin >= k".
# The cell that fits the threshold asserts that rather than trusting it.
#
# 0.5% is deliberate. The bootstrap interval on tau* is about +-5 points and
# the plateau spans tens of points, so a finer grid would only manufacture
# precision the data cannot support. It is not 1% because the DWD threshold of
# 12.5% has to be exactly representable. tau* is *reported* to the nearest
# whole percent throughout, which is already finer than the data warrants.
BIN_PER_PCT = 2
NBIN = 100 * BIN_PER_PCT + 1
TAUS = np.arange(NBIN) / BIN_PER_PCT

# --- corrections ------------------------------------------------------------
PARALLAX = True          # displace the mask onto true ground positions
# Verified 2026-09-03 against the files themselves; see docs/conventions.md.
# The NWCSAF slot label is the nominal START of the SEVIRI repeat cycle, this
# domain is imaged +11.2 min into it, and the model writes hourly only. So the
# standard pairing is the slot labelled T-15min against the model field valid
# at T, leaving a residual of -3.8 min (observation slightly before the model).
#
# Applied uniformly, with no fallback: an hour whose T-15 slot is missing is
# DROPPED rather than quietly paired with a different slot. Mixing conventions
# inside one verification period is exactly the failure this guards against.
NWCSAF_SLOT_LAG = timedelta(minutes=15)

# Cloud types dropped from the comparison. Only snow/ice: there the mask may be
# genuinely WRONG (a bright surface read as cloud, or cloud lost against it),
# which is observation error. Fractional and thin-cirrus pixels are deliberately
# KEPT -- the satellite really did detect them, the pixel really is partly
# cloudy, and since CLCT is itself a partial-cloud fraction those pixels are
# exactly where the continuous-vs-binary mismatch lives. Dropping them would
# discard the hardest and most informative cases and leave a cloud-depleted
# sample.
CT_SCREEN = "snow_ice"      # "snow_ice" | "snow_ice+fractional+thin" | "none"
SAT_LON = 0.0            # Meteosat-10 sub-satellite longitude

# Elevation bands for the terrain stratification (m).
ELEV_EDGES = [-100, 800, 1500, 5000]

# Statistics
DECORR_H = 4             # block length for the bootstrap; re-measured in sec. 8
N_BOOT = 1000
MIN_EFFECTIVE_N = 30

print(f"exp {EXP}: {len(TIMES)} hourly steps, {TIMES[0]} .. {TIMES[-1]}")

## 1. Conservative regridding (ICON native mesh -> NWCSAF grid)

Unchanged from `CLCT_binary_verification.ipynb`, and the weights cache is shared with it: every ICON
triangle is clipped against every NWCSAF cell it overlaps and weighted by the exact overlap area, so
`remap()` is two `bincount` calls. `check_partition_of_unity()` re-validates on every run, cache or not.

The one thing worth restating: the target grid's latitude axis **descends**, so `dlat` is negative
everywhere below. Absolute values are used for cell sizes and the signed value for index arithmetic --
getting that backwards would flip the parallax displacement north/south, which is exactly the error this
notebook is trying to remove.

In [ ]:
def read_grid(path, domain=None):
    """1-D coordinates of the NWCSAF grid, validated as regular."""
    with xr.open_dataset(path) as ds:
        lon2 = np.asarray(ds["longitude"].values)
        lat2 = np.asarray(ds["latitude"].values)

    if lon2.ndim == 2:
        if (np.abs(lon2 - lon2[0, :]).max() > 1e-9
                or np.abs(lat2 - lat2[:, [0]]).max() > 1e-9):
            raise ValueError("2-D coordinates are not a regular lat/lon grid")
        lon, lat = lon2[0, :], lat2[:, 0]
    else:
        lon, lat = lon2, lat2

    dlon = float(np.median(np.diff(lon)))
    dlat = float(np.median(np.diff(lat)))     # negative: latitude descends
    if (np.abs(np.diff(lon) - dlon).max() > 1e-9
            or np.abs(np.diff(lat) - dlat).max() > 1e-9):
        raise ValueError("grid spacing is not uniform")

    if domain is not None:
        jx = np.where((lon >= domain[0]) & (lon <= domain[1]))[0]
        iy = np.where((lat >= domain[2]) & (lat <= domain[3]))[0]
        lon, lat = lon[jx], lat[iy]
    return lon, lat, dlon, dlat


def read_icon_cells(grid_file, n_expected, lon_c, lat_c):
    """ICON triangle vertices in degrees, validated against the field."""
    with xr.open_dataset(grid_file) as g:
        vlon = np.asarray(g["clon_vertices"].values, dtype="float64")
        vlat = np.asarray(g["clat_vertices"].values, dtype="float64")
    if vlon.shape[0] != n_expected:
        raise ValueError(f"grid file has {vlon.shape[0]} cells, field has {n_expected}")
    if np.abs(vlat).max() <= np.pi + 1e-6:      # no units attribute; detect radians
        vlon, vlat = np.rad2deg(vlon), np.rad2deg(vlat)
    dv = max(np.abs(vlon.mean(axis=1) - lon_c).max(),
             np.abs(vlat.mean(axis=1) - lat_c).max())
    if dv > 1e-3:
        raise ValueError("vertices do not match the field's cell centres")
    return vlon, vlat


def triangle_polygons(vlon, vlat, sub):
    rings = np.stack([vlon[sub], vlat[sub]], axis=-1)
    rings = np.concatenate([rings, rings[:, :1, :]], axis=1)
    return shapely.polygons(rings)


def build_weights(lon, lat, dlon, dlat, vlon, vlat):
    ny, nx = lat.size, lon.size
    ncell = ny * nx
    hx, hy = abs(dlon) / 2, abs(dlat) / 2

    near = ((vlon.max(axis=1) >= lon.min() - hx)
            & (vlon.min(axis=1) <= lon.max() + hx)
            & (vlat.max(axis=1) >= lat.min() - hy)
            & (vlat.min(axis=1) <= lat.max() + hy))
    sub = np.where(near)[0]
    tri = triangle_polygons(vlon, vlat, sub)

    LON, LAT = np.meshgrid(lon, lat)
    cells = shapely.box((LON - hx).ravel(), (LAT - hy).ravel(),
                        (LON + hx).ravel(), (LAT + hy).ravel())

    ti, ci = shapely.STRtree(cells).query(tri, predicate="intersects")
    w = np.empty(ti.size)
    for a in range(0, ti.size, CHUNK):
        b = min(a + CHUNK, ti.size)
        w[a:b] = shapely.area(shapely.intersection(tri[ti[a:b]], cells[ci[a:b]]))
    keep = w > 0
    ti, ci, w = ti[keep], ci[keep], w[keep]

    coverage = np.bincount(ci, weights=w, minlength=ncell) / (abs(dlon) * abs(dlat))
    print(f"target cells >= {MIN_COVERAGE} covered: {(coverage >= MIN_COVERAGE).sum()}/{ncell}")
    return sub, ti, ci, w, coverage.reshape(ny, nx)


def check_partition_of_unity(sub, ti, w, vlon, vlat, lon, lat, dlon, dlat, tol=1e-9):
    """Each interior triangle's area must be exactly partitioned among the cells
    it overlaps. Restricted to triangles fully inside the target grid -- edge
    triangles legitimately lose area, and triangles selected only by their
    bounding box contribute none, so both would pass trivially."""
    hx, hy = abs(dlon) / 2, abs(dlat) / 2
    want = shapely.area(triangle_polygons(vlon, vlat, sub))
    got = np.bincount(ti, weights=w, minlength=sub.size)
    interior = ((vlon[sub].min(axis=1) > lon.min() - hx)
                & (vlon[sub].max(axis=1) < lon.max() + hx)
                & (vlat[sub].min(axis=1) > lat.min() - hy)
                & (vlat[sub].max(axis=1) < lat.max() + hy))
    err = float(np.abs(got[interior] / want[interior] - 1).max())
    print(f"partition of unity ({int(interior.sum())} interior triangles of {sub.size} "
          f"selected): max |sum(overlap)/area - 1| = {err:.2e}")
    if err > tol:
        raise RuntimeError("regridding weights do not partition triangle area")
    return err


def remap(values, sub, ti, ci, w, coverage, shape):
    v = np.asarray(values, dtype="float64").ravel()[sub]
    ok = np.isfinite(v[ti])
    ncell = shape[0] * shape[1]
    num = np.bincount(ci[ok], weights=w[ok] * v[ti][ok], minlength=ncell)
    den = np.bincount(ci[ok], weights=w[ok], minlength=ncell)
    out = np.full(ncell, np.nan)
    good = (coverage.ravel() >= MIN_COVERAGE) & (den > 0)
    out[good] = num[good] / den[good]
    return out.reshape(shape)

In [ ]:
# --- grids and weights ------------------------------------------------------
probe_time = TIMES[0]
lon, lat, dlon, dlat = read_grid(nwcsaf_file(probe_time), DOMAIN)
shape = (lat.size, lon.size)
CELL_KM_X = abs(dlon) * 111.32 * np.cos(np.deg2rad(float(np.mean(lat))))
CELL_KM_Y = abs(dlat) * 111.32
print(f"target grid {shape}, cell {CELL_KM_X:.2f} x {CELL_KM_Y:.2f} km, "
      f"dlon {dlon:+.5f}, dlat {dlat:+.5f} "
      f"(latitude {'descends' if dlat < 0 else 'ascends'})")

fl = ekd.from_source("file", icon_file(probe_time)).to_fieldlist()
xa = fl.sel({"parameter.variable": "CLCT"})[0].to_xarray()
name0 = "CLCT" if "CLCT" in xa else list(xa.data_vars)[0]
lon_c = np.asarray(xa["longitude"].values).ravel()
lat_c = np.asarray(xa["latitude"].values).ravel()
vlon, vlat = read_icon_cells(ICON_GRID_FILE, lon_c.size, lon_c, lat_c)

if WEIGHTS_CACHE and os.path.exists(WEIGHTS_CACHE):
    z = np.load(WEIGHTS_CACHE)
    sub, ti, ci, w, coverage = z["sub"], z["ti"], z["ci"], z["w"], z["coverage"]
    print(f"weights loaded from {WEIGHTS_CACHE}")
else:
    sub, ti, ci, w, coverage = build_weights(lon, lat, dlon, dlat, vlon, vlat)
    if WEIGHTS_CACHE:
        np.savez_compressed(WEIGHTS_CACHE, sub=sub, ti=ti, ci=ci, w=w, coverage=coverage)
        print(f"weights saved to {WEIGHTS_CACHE}")

check_partition_of_unity(sub, ti, w, vlon, vlat, lon, lat, dlon, dlat)
in_domain = coverage >= MIN_COVERAGE

# --- static model orography, on the target grid ------------------------------
hs = fl.sel({"parameter.variable": "HSURF"})
if len(hs) == 0:
    raise RuntimeError("HSURF not in the lff file -- needed for terrain parallax "
                       "and the elevation stratification")
xh = hs[0].to_xarray()
nmh = "HSURF" if "HSURF" in xh else list(xh.data_vars)[0]
HSURF = remap(np.asarray(xh[nmh].values, dtype="float64").ravel(),
              sub, ti, ci, w, coverage, shape)
HSURF_KM = np.where(np.isfinite(HSURF), HSURF, 0.0) / 1000.0

ELEV_BAND = np.full(shape, -1, dtype=int)
elev_names = []
for k in range(len(ELEV_EDGES) - 1):
    lo_e, hi_e = ELEV_EDGES[k], ELEV_EDGES[k + 1]
    ELEV_BAND[np.isfinite(HSURF) & (HSURF >= lo_e) & (HSURF < hi_e)] = k
    elev_names.append(f"{lo_e}-{hi_e} m")

print(f"\n{int(in_domain.sum())} cells inside the ICON domain")
print("elevation bands (cells inside the domain):")
for k, nm in enumerate(elev_names):
    print(f"  {nm:12s} {int(((ELEV_BAND == k) & in_domain).sum()):6d}")

## 2. Viewing geometry: parallax displacement and scan timing

Both corrections come out of the same geostationary geometry, so they are derived together and per pixel
rather than for the domain centre only.

**Displacement.** For a pixel at `(lat, lon)` seen from a satellite at `SAT_LON`, the viewing zenith angle
follows from the angular distance `psi` to the sub-satellite point. A feature at height `z` is reported
`tan(vza) * z` further from the sub-satellite point than it really is, so the correction moves it
`tan(vza) * z` *towards* the sub-satellite point, along the great-circle bearing to it. Over this domain
`tan(vza)` is around 1.4, and the direction is essentially due south with a small westward component.

**Scan timing.** SEVIRI acquires the full disc south to north. The files themselves carry `start_time` and
`end_time` on every variable (12m23s apart), so the acquisition window is read rather than assumed; only
the *fraction* of it at which this domain is imaged is computed, from the north-south geostationary angle
`y` and the disc half-extent. Section 5 checks the result against those attributes.

In [ ]:
R_E, R_S = 6371.0, 42164.0          # km: Earth radius, geostationary orbit radius
Y_DISC = 0.1518                     # rad: half the north-south angular extent of the disc

LON2, LAT2 = np.meshgrid(lon, lat)
_la, _dlo = np.deg2rad(LAT2), np.deg2rad(LON2 - SAT_LON)

PSI = np.arccos(np.cos(_la) * np.cos(_dlo))                 # angular distance to sub-sat point
VZA = np.arctan2(R_S * np.sin(PSI), R_S * np.cos(PSI) - R_E)
# initial great-circle bearing from each pixel towards the sub-satellite point
BRG = np.arctan2(np.sin(-_dlo), -np.sin(_la) * np.cos(-_dlo))

# displacement per km of feature height, in degrees, towards the sub-satellite point
DLAT_PER_KM = np.tan(VZA) * np.cos(BRG) / 111.32
DLON_PER_KM = np.tan(VZA) * np.sin(BRG) / (111.32 * np.cos(_la))

_c = (shape[0] // 2, shape[1] // 2)
print(f"domain centre {LAT2[_c]:.2f}N {LON2[_c]:.2f}E: viewing zenith "
      f"{np.rad2deg(VZA[_c]):.1f} deg, displacement {np.tan(VZA[_c]):.2f} x height")
print(f"viewing zenith across the domain: {np.rad2deg(VZA).min():.1f} .. "
      f"{np.rad2deg(VZA).max():.1f} deg")
print("\nshift applied to the observation, at the domain centre:")
for z in (0.5, 1, 2, 5, 8):
    print(f"  height {z:4.1f} km -> {z * DLAT_PER_KM[_c] / dlat:+5.2f} rows "
          f"({z * DLAT_PER_KM[_c] * 111.32:+5.1f} km north), "
          f"{z * DLON_PER_KM[_c] / dlon:+5.2f} cols")

# --- scan fraction: where in the south-to-north sweep this domain is imaged ---
def geos_y_angle(lat_deg, lon_deg, sat_lon=SAT_LON):
    """North-south geostationary viewing angle. Positive south, negative north."""
    c_lat = np.arctan(0.993243 * np.tan(np.deg2rad(lat_deg)))
    r_l = 6356.5838 / np.sqrt(1 - 0.00669438 * np.cos(c_lat) ** 2)
    dl = np.deg2rad(lon_deg - sat_lon)
    r1 = 42164.0 - r_l * np.cos(c_lat) * np.cos(dl)
    r2 = -r_l * np.cos(c_lat) * np.sin(dl)
    r3 = r_l * np.sin(c_lat)
    rn = np.sqrt(r1 ** 2 + r2 ** 2 + r3 ** 2)
    return np.arcsin(-r3 / rn)


lat0, lon0 = float(np.mean(lat)), float(np.mean(lon))
y_ang = geos_y_angle(lat0, lon0)
SCAN_FRAC = float((Y_DISC - y_ang) / (2 * Y_DISC))   # 0 at the south edge, 1 at the north
print(f"\ndomain centre is imaged {100 * SCAN_FRAC:.0f}% of the way through the "
      f"south-to-north sweep")

## 3. What the NWCSAF files contain

These exports carry `cma`, `ct`, `ctth_alti/pres/tempe` and the CTTH flag variables. **There is no
`cma_quality` or `cma_conditions`** — the cloud mask arrives as a bare 0/1 field with no confidence
attached, so it cannot be screened by its own quality. Two consequences:

- the `ctth_*` flags describe the *height* retrieval, so they gate the parallax correction, not the mask;
- screening of the mask itself goes through `ct` (cloud type), which is also where the illumination
  (`ctth_conditions`: day / night / twilight) comes from, replacing a hand-rolled solar zenith angle.

Flag layouts are decoded from each file's own `flag_values`/`flag_mask` attributes rather than
hard-coded. Details of what was found are in the NWCSAF file-internals notes.

In [ ]:
def decode_flags(da):
    """Named boolean layers from a CF flag variable, using the file's own
    flag_values/flag_mask attributes rather than a hard-coded bit layout.
    Duplicate flag_meanings (the files contain two 'not_used') get suffixed."""
    a = np.asarray(da.values)
    ai = np.where(np.isfinite(a), a, 0).astype("int64")
    meanings = da.attrs["flag_meanings"].split()
    values = np.asarray(da.attrs["flag_values"]).astype("int64")
    masks = da.attrs.get("flag_mask", da.attrs.get("flag_masks", values))
    masks = np.asarray(masks).astype("int64")
    out, seen = {}, {}
    for m, v, mk in zip(meanings, values, masks):
        seen[m] = seen.get(m, 0) + 1
        key = m if seen[m] == 1 else f"{m}__{seen[m]}"
        out[key] = (ai & int(mk)) == int(v)
    return out


CT_SNOW_ICE = (3, 4)          # Snow_over_land, Sea_ice
CT_FRACTIONAL = (10,)         # Fractional_clouds -- sub-pixel
CT_THIN = (11,)               # High_semitransparent_thin_clouds
CT_SEMITRANSP = (11, 12, 13, 14, 15)

SCREEN_SETS = {"none": (),
               "snow_ice": CT_SNOW_ICE,
               "snow_ice+fractional+thin": CT_SNOW_ICE + CT_FRACTIONAL + CT_THIN}
SCREEN_CLASSES = SCREEN_SETS[CT_SCREEN]


def read_nwcsaf(path, domain=None):
    """Cloud mask plus everything needed to correct and screen it."""
    with xr.open_dataset(path) as ds:
        if domain is not None:
            lo1 = np.asarray(ds["longitude"].values)
            la1 = np.asarray(ds["latitude"].values)
            lo1 = lo1[0, :] if lo1.ndim == 2 else lo1
            la1 = la1[:, 0] if la1.ndim == 2 else la1
            jx = np.where((lo1 >= domain[0]) & (lo1 <= domain[1]))[0]
            iy = np.where((la1 >= domain[2]) & (la1 <= domain[3]))[0]
            sl = np.ix_(iy, jx)
        else:
            sl = (slice(None), slice(None))

        cma = np.asarray(ds["cma"].squeeze().values, dtype="float64")[sl]
        cma[cma == 255] = np.nan                       # no-op in these exports; kept defensively
        ct = np.asarray(ds["ct"].squeeze().values, dtype="float64")[sl]
        alti = np.asarray(ds["ctth_alti"].squeeze().values, dtype="float64")[sl]
        lo, hi = ds["ctth_alti"].attrs.get("valid_range", (-2000.0, 25000.0))
        alti[(alti < lo) | (alti > hi)] = np.nan

        qual = decode_flags(ds["ctth_quality"])
        cond = decode_flags(ds["ctth_conditions"])
        meth = decode_flags(ds["ctth_method"])
        stat = decode_flags(ds["ctth_status_flag"])
        qual = {k: v[sl] for k, v in qual.items()}
        cond = {k: v[sl] for k, v in cond.items()}
        meth = {k: v[sl] for k, v in meth.items()}
        stat = {k: v[sl] for k, v in stat.items()}

        t0 = str(ds["cma"].attrs.get("start_time", ""))
        t1 = str(ds["cma"].attrs.get("end_time", ""))

    return dict(cma=cma, ct=ct, alti=alti, qual=qual, cond=cond, meth=meth,
                stat=stat, start_time=t0, end_time=t1)


# --- one-line inventory, on the probe slot -------------------------------
S = read_nwcsaf(nwcsaf_file(datetime(2025, 10, 6, 11)), DOMAIN)
_cl = S["cma"] == 1
_h = _cl & np.isfinite(S["alti"])
print(f"probe slot: {int(_cl.sum())} cloudy / {int((S['cma'] == 0).sum())} clear; "
      f"{100 * _h.sum() / max(_cl.sum(), 1):.0f}% of cloudy pixels have a usable "
      f"ctth_alti\n(the rest cannot be parallax-corrected); screening set "
      f"'{CT_SCREEN}' -> ct classes {SCREEN_CLASSES}")


## 4. Parallax correction of the mask

Each pixel is moved from where MSG reports it to the ground point it actually sits over, and the corrected
mask is rebuilt by scattering pixels into the cells they land in:

- **Cloudy pixels with a usable `ctth_alti`** move by `tan(vza) * z` towards the sub-satellite point.
- **Clear pixels** move by `tan(vza) * HSURF`. This is not a refinement to be skipped: at 2.5 km of Alpine
  terrain the displacement is around 3.5 km, a full grid cell, and it is systematic rather than random.
- **Cloudy pixels without a height retrieval** (about 9% of cloudy pixels, `ctth_method` = no reliable
  method) cannot be placed at all and become NaN. Guessing a height for them would manufacture exactly the
  signal being measured.

Two things then happen that a "shift the field" implementation hides, and both are reported per timestep:

- **Gaps.** A ground cell can end up receiving no pixel at all -- it was hidden behind a displaced cloud.
  It is genuinely unobserved and stays NaN rather than being filled with the nearest value.
- **Collisions.** A cell can receive both a displaced cloudy and a displaced clear pixel. Cloud wins: if
  any line of sight over that ground point held cloud, the column is cloudy.

`ctth_quality` gates the correction rather than the mask -- a `bad` height would displace a real cloud to
the wrong place, which is worse than not moving it. Those pixels are treated like the ones with no height
at all. This is a deliberately conservative choice and it costs sample size; section 9 shows how much.

In [ ]:
def parallax_correct(sat, hsurf_km, strict_quality=False):
    """Displace the mask onto true ground positions. Returns the corrected mask
    and a diagnostics dict.

    `strict_quality` also discards heights flagged `bad` by ctth_quality. It is
    off by default, and the reason is worth stating: not displacing a cloudy
    pixel is not a neutral act, it is equivalent to asserting the cloud top is
    at height zero. For a pixel the mask calls cloudy that is the one height it
    certainly is not. A `bad` retrieval is a poor estimate; no retrieval is a
    guaranteed-wrong one. In these files `bad` covers 28% of pixels, so strict
    gating leaves about a third of the domain unobserved -- the ladder in
    section 9 carries both variants so the trade is visible rather than
    assumed."""
    cma = sat["cma"]
    ny, nx = cma.shape
    cloudy = cma == 1
    clear = cma == 0

    h_km = sat["alti"] / 1000.0
    ok_h = np.isfinite(h_km)
    ok_h &= ~sat["meth"].get("No_reliable_method", np.zeros_like(ok_h))
    if strict_quality:
        ok_h &= ~sat["qual"].get("bad", np.zeros_like(ok_h))

    movable = cloudy & ok_h
    lost = cloudy & ~ok_h

    def scatter(mask, z_km):
        i0, j0 = np.nonzero(mask)
        lat_t = LAT2[i0, j0] + DLAT_PER_KM[i0, j0] * z_km[i0, j0]
        lon_t = LON2[i0, j0] + DLON_PER_KM[i0, j0] * z_km[i0, j0]
        i1 = np.rint((lat_t - lat[0]) / dlat).astype(int)
        j1 = np.rint((lon_t - lon[0]) / dlon).astype(int)
        keep = (i1 >= 0) & (i1 < ny) & (j1 >= 0) & (j1 < nx)
        cnt = np.zeros((ny, nx), dtype=np.int32)
        np.add.at(cnt, (i1[keep], j1[keep]), 1)
        return cnt, int((~keep).sum())

    n_cloud, off_c = scatter(movable, np.where(ok_h, h_km, 0.0))
    n_clear, off_k = scatter(clear, hsurf_km)

    out = np.full((ny, nx), np.nan)
    out[n_clear > 0] = 0.0
    out[n_cloud > 0] = 1.0                       # cloud wins a collision

    # A cell that an uncorrectable cloudy pixel could belong to, and where no
    # confirmed cloud landed, is unknown -- even if a displaced clear pixel also
    # landed there, because nothing rules cloud out. The earlier form of this
    # line also required n_clear == 0, which made it dead code: with no cloud
    # and no clear pixel the cell is already NaN from the initialisation above.
    # The case it was meant to catch (lost cloud + a clear pixel -> marked
    # CLEAR) therefore leaked through. Measured at 0.03% of cells, so this
    # changes no result, but the assertion it encodes is now the one intended.
    n_lost, _ = scatter(lost, hsurf_km)
    out[(n_lost > 0) & (n_cloud == 0)] = np.nan

    valid_before = np.isfinite(cma) & in_domain
    valid_after = np.isfinite(out) & in_domain
    diag = dict(
        n_movable=int(movable.sum()), n_lost=int(lost.sum()),
        gap_frac=float(1 - valid_after.sum() / max(valid_before.sum(), 1)),
        collision=int(((n_cloud > 0) & (n_clear > 0) & in_domain).sum()),
        off_grid=off_c + off_k,
        changed=float(np.nanmean((out != cma)[valid_before & valid_after])),
    )
    return out, diag

## 5. Which satellite slot matches an ICON hour

**This is now a verified convention, not a heuristic. See `docs/conventions.md`.**

`lff` CLCT is instantaneous and valid exactly on the hour. The satellite slot named `HH:00` is not an
observation at `HH:00`: the slot label is the nominal *start* of the SEVIRI repeat cycle, confirmed on
six slots spread across the period, where `start_time` equals the filename label exactly and
`end_time = start + 743 s` (12.38 min -- a full-disk scan, not a subregion window). SEVIRI scans south
to north, and this domain sits about 90% of the way through the sweep, so it is imaged **+11.2 min**
after the label, with only 8 seconds of spread between 45.8N and 47.8N.

Model output for these experiments is hourly only, so a slot must be chosen rather than interpolated,
and CMA is categorical, so interpolating between model fields would be wrong anyway. The standard is
therefore:

> **NWCSAF slot `T-15min`  <->  model field valid at `T`**, leaving a residual of **-3.8 min**
> (observation slightly before the model) instead of **+11.2 min** the other way.

It is applied **uniformly and without fallback**. An hour whose `T-15` slot is missing is dropped, not
paired with a neighbouring slot: keeping the hour would mean scoring it against an observation on the
opposite side of the model time, which is precisely the convention-mixing this rule exists to prevent.
Two model hours at the ends of the window are lost that way, which is the intended trade.

**What it buys, stated honestly.** Measured over 144 hours, changing only the paired slot moved pooled
`max J` from 0.5002 to 0.5047 -- about **+0.004**, with the matched slot better in 59% of hours. A naive
significance test on those hours is misleading because they are not independent (the domain-mean bias
decorrelates over ~5 h, so the effective sample is ~29), and on that basis the gain is roughly 1.4
standard errors: not distinguishable from zero. The convention is adopted because it is physically
correct and cleanly documentable, **not** because it improves scores -- the `ct` screening and parallax
correction below are each an order of magnitude larger. It should not be cited downstream as a skill
improvement.

The offset is derived at runtime from each file's own `start_time`/`end_time` and the domain's
geostationary scan angle rather than hard-coded, so it self-corrects if the scan duration, domain or
satellite ever changes.

In [ ]:
# acquisition duration straight from the files
_probe = read_nwcsaf(nwcsaf_file(datetime(2025, 10, 6, 11)), DOMAIN)
_t0 = datetime.fromisoformat(_probe["start_time"])
_t1 = datetime.fromisoformat(_probe["end_time"])
SCAN_DUR_S = (_t1 - _t0).total_seconds()
OFFSET = timedelta(seconds=SCAN_FRAC * SCAN_DUR_S)
print(f"acquisition {_t0:%H:%M:%S} .. {_t1:%H:%M:%S}  ->  {SCAN_DUR_S:.0f} s "
      f"({SCAN_DUR_S / 60:.1f} min)")
print(f"this domain is imaged {OFFSET.total_seconds() / 60:.1f} min after the "
      f"nominal slot time\n")


def match_slot(t):
    """The NWCSAF slot paired with model valid time `t`.

    Strict by design: the slot is always `t - NWCSAF_SLOT_LAG`, and if that file
    is missing the hour is dropped. Searching neighbouring slots would keep more
    hours at the cost of scoring some of them against an observation 11 min on
    the other side of the model time, which is the convention-mixing the
    verification explicitly warned against."""
    slot = t - NWCSAF_SLOT_LAG
    return slot if os.path.exists(nwcsaf_file(slot)) else None


def acquisition_error_min(slot, t):
    """Signed minutes between when this domain is imaged and the model time."""
    return (slot + OFFSET - t).total_seconds() / 60


print(f"convention: slot T-{int(NWCSAF_SLOT_LAG.total_seconds()//60)}min <-> "
      f"model valid T   (residual "
      f"{acquisition_error_min(TIMES[0] - NWCSAF_SLOT_LAG, TIMES[0]):+.1f} min)\n")
print(f"{'ICON hour':<17s} {'paired slot':>12s} {'imaged at':>11s} {'residual':>9s}"
      f"   {'superseded slot T':>18s} {'residual':>9s}")
for t in [TIMES[0], datetime(2025, 10, 6, 11), datetime(2025, 10, 6, 12), TIMES[-1]]:
    sl = match_slot(t)
    nom = os.path.exists(nwcsaf_file(t))
    print(f"{t:%Y-%m-%d %H:%M} {(sl.strftime('%H:%M') if sl else 'MISSING'):>12s} "
          f"{((sl + OFFSET).strftime('%H:%M:%S') if sl else '--'):>11s} "
          f"{(acquisition_error_min(sl, t) if sl else float('nan')):+8.1f}m   "
          f"{(t.strftime('%H:%M') if nom else 'MISSING'):>18s} "
          f"{(acquisition_error_min(t, t) if nom else float('nan')):+8.1f}m")

# Orphans, stated up front rather than discovered as a silent shortfall.
_orphan_hours = [t for t in TIMES if match_slot(t) is None]
_used = {match_slot(t) for t in TIMES if match_slot(t) is not None}
_all_slots = []
_s = TIMES[0] - NWCSAF_SLOT_LAG
while _s <= TIMES[-1]:
    if os.path.exists(nwcsaf_file(_s)):
        _all_slots.append(_s)
    _s += timedelta(minutes=15)
print(f"\nmodel hours with no T-15 slot (dropped): {len(_orphan_hours)}"
      + (f" -> {', '.join(f'{t:%m-%d %Hh}' for t in _orphan_hours[:6])}"
         if _orphan_hours else ""))
print(f"satellite slots in range never paired (hourly model, 15-min satellite): "
      f"{len(_all_slots) - len(_used)} of {len(_all_slots)}")

## 6. Main loop: from fields to histograms

The threshold sweep, the bootstrap, the cross-validation and every stratum all reduce to counting how many
cloudy and how many clear pixels fall in each CLCT bin. So the loop stores exactly that -- a pair of
`(n_time, 201)` count arrays per configuration and per stratum -- and never keeps the fields. Every J
evaluation afterwards is a sum over precomputed counts, which is what makes a thousand bootstrap replicates
across a dozen configurations cheap rather than an overnight job.

Configurations built in one pass:

| forecast | observation | slot |
|---|---|---|
| CLCT | mask as delivered | `T-15` **(baseline)** |
| CLCT | + `ct` screening | `T-15` |
| CLCT | + parallax | `T-15` |
| CLCT | + parallax, strict ctth quality | `T-15` |
| detectability-filtered CLCT | + parallax | `T-15` |
| CLCT | mask as delivered | `T` *(superseded reference only)* |

Every configuration now pairs the model hour with the verified `T-15` slot; the timing convention is no
longer a variable in the comparison. The last row keeps the old slot-`T` pairing purely so section 9
can show what the superseded convention cost.

Strata (`all`, elevation band, illumination, cloud class) are accumulated alongside, so section 11 needs no
second pass.

In [ ]:
def to_bins(clct):
    """Bin index such that (CLCT >= TAUS[k]) is exactly (bin >= k)."""
    v = np.where(np.isfinite(clct), clct, 0.0)
    return np.clip(np.floor(v * BIN_PER_PCT), 0, NBIN - 1).astype(int)


def hist_pair(clct, obs, valid):
    """Per-bin counts of cloudy and clear pixels."""
    m = valid & np.isfinite(clct) & np.isfinite(obs)
    b = to_bins(clct)[m]
    o = obs[m] == 1
    return (np.bincount(b[o], minlength=NBIN).astype(np.int64),
            np.bincount(b[~o], minlength=NBIN).astype(np.int64))


# Every config pairs the model hour with the verified T-15 slot; timing is no
# longer a variable. Two earlier diagnostic configs (strict ctth quality, and
# the superseded slot-T pairing) have been retired -- both were measured, both
# are recorded in docs/conventions.md, and neither is used downstream.
CONFIGS = ["scan", "scan+ct", "scan+ct+plx"]
STRATA = ["all"] + [f"elev:{n}" for n in elev_names] + \
         ["illum:day", "illum:night", "illum:twilight",
          "class:opaque", "class:fractional", "class:thin", "class:snow_ice"]

# A stale cache silently reused would be a nasty bug, so the signature records
# everything that changes what the histograms mean. Any mismatch rebuilds.
HIST_SIGNATURE = "|".join([EXP, str(START), str(N_HOURS), str(SPLIT_DATE),
                           str(DOMAIN), str(MIN_COVERAGE), str(BIN_PER_PCT),
                           str(PARALLAX), str(NWCSAF_SLOT_LAG),
                           ",".join(CONFIGS), ",".join(STRATA)])

# Two filesystem stalls in one session each cost a full hour of loop, because
# the cache was only written at the very end. It is now written every
# HIST_CHECKPOINT_EVERY hours and the loop resumes from the last checkpoint.
HIST_CHECKPOINT_EVERY = 12


def _save_hist(lists, times_done, complete):
    if not HIST_CACHE or not times_done:
        return
    np.savez_compressed(
        HIST_CACHE,
        **{f"{c}|{st}|{k}": np.stack(v) for (c, st, k), v in lists.items()},
        times=np.array([x.isoformat() for x in times_done]),
        signature=np.array(HIST_SIGNATURE),
        complete=np.array(bool(complete)))


_cached = None
_resume = None
if HIST_CACHE and os.path.exists(HIST_CACHE):
    _z = np.load(HIST_CACHE, allow_pickle=False)
    _ok = "signature" in _z.files and str(_z["signature"]) == HIST_SIGNATURE
    _done_flag = bool(_z["complete"]) if "complete" in _z.files else True
    if _ok and _done_flag:
        _cached = _z
        print(f"histograms loaded from {HIST_CACHE} (signature matches)")
    elif _ok:
        _resume = _z
        print(f"partial cache: resuming after {len(_z['times'])} of "
              f"{len(TIMES)} hours")
    else:
        print(f"cache at {HIST_CACHE} was built with different settings -- rebuilding")

if _cached is not None:
    TIMES_used = [datetime.fromisoformat(str(x)) for x in _cached["times"]]
    H = {tuple(k.split("|")): _cached[k] for k in _cached.files
         if k not in ("times", "signature")}
    NT = len(TIMES_used)
    slot_used, diag_rows = [], []
    print(f"{NT} hours, {TIMES_used[0]} .. {TIMES_used[-1]}")
    print("(set HIST_CACHE = None, or delete the file, to force a re-read of the fields)")
else:
  H = {(c, s, k): [] for c in CONFIGS for s in STRATA for k in ("cloud", "clear")}
  TIMES_used, slot_used, diag_rows = [], [], []

  if _resume is not None:
      TIMES_used = [datetime.fromisoformat(str(x)) for x in _resume["times"]]
      for _key in H:
          H[_key] = list(_resume["|".join(_key)])
  _done = set(TIMES_used)

  for _i_t, t in enumerate(TIMES, 1):
      if t in _done:
          continue
      try:
          fl_t = ekd.from_source("file", icon_file(t)).to_fieldlist()
          sel = fl_t.sel({"parameter.variable": "CLCT"})
          if len(sel) == 0:
              raise FileNotFoundError("no CLCT field")
          xa_t = sel[0].to_xarray()
          nm = "CLCT" if "CLCT" in xa_t else list(xa_t.data_vars)[0]
          clct = remap(np.asarray(xa_t[nm].values, dtype="float64").ravel(),
                       sub, ti, ci, w, coverage, shape)

          slot = match_slot(t)
          if slot is None:
              raise FileNotFoundError(
                  "no T-15 slot for this hour; dropped rather than paired with "
                  "a different slot (see docs/conventions.md)")
          Ss = read_nwcsaf(nwcsaf_file(slot), DOMAIN)
      except Exception as e:
          print(f"  skip {t}: {type(e).__name__}: {e}")
          continue

      ct = Ss["ct"]
      screen = ~np.isin(ct, SCREEN_CLASSES) if SCREEN_CLASSES else np.ones(shape, bool)
      cma_plx, diag = (parallax_correct(Ss, HSURF_KM) if PARALLAX
                       else (Ss["cma"], {}))


      variants = {
            "scan": (clct, Ss["cma"], in_domain),
          "scan+ct": (clct, Ss["cma"], in_domain & screen),
          "scan+ct+plx": (clct, cma_plx, in_domain & screen),
          }

      strat_masks = {"all": np.ones(shape, dtype=bool)}
      for k_, n_ in enumerate(elev_names):
          strat_masks[f"elev:{n_}"] = ELEV_BAND == k_
      for nm_, key in (("day", "day"), ("night", "night"), ("twilight", "twilight")):
          strat_masks[f"illum:{nm_}"] = Ss["cond"].get(key, np.zeros(shape, bool))
      strat_masks["class:opaque"] = np.isin(ct, (5, 6, 7, 8, 9))
      strat_masks["class:fractional"] = np.isin(ct, CT_FRACTIONAL)
      strat_masks["class:thin"] = np.isin(ct, CT_THIN)
      strat_masks["class:snow_ice"] = np.isin(ct, CT_SNOW_ICE)

      for cfg, (fc, ob, base) in variants.items():
          for s_ in STRATA:
              if fc is None or ob is None:
                  hc = hk = np.zeros(NBIN, dtype=np.int64)
              else:
                  hc, hk = hist_pair(fc, ob, base & strat_masks[s_])
              H[(cfg, s_, "cloud")].append(hc)
              H[(cfg, s_, "clear")].append(hk)

      TIMES_used.append(t)
      slot_used.append(slot)
      diag_rows.append(diag)

      # An hour-long cell that prints nothing is indistinguishable from a hung
      # one -- which is exactly how the last stall went unnoticed for 15 min.
      if len(TIMES_used) % HIST_CHECKPOINT_EVERY == 0:
          _save_hist(H, TIMES_used, complete=False)
          print(f"  {_i_t}/{len(TIMES)} hours ({t:%m-%d %Hh}) -- checkpointed",
                flush=True)

  _save_hist(H, TIMES_used, complete=True)
  H = {k: np.stack(v) for k, v in H.items()}
  NT = len(TIMES_used)
  print(f"\n{NT}/{len(TIMES)} hours usable, {TIMES_used[0]} .. {TIMES_used[-1]}")
  if PARALLAX and diag_rows and diag_rows[0]:
      print(f"parallax, averaged over the period: "
            f"{100 * np.mean([d['gap_frac'] for d in diag_rows]):.1f}% of cells left "
            f"unobserved, {100 * np.mean([d['changed'] for d in diag_rows]):.1f}% "
            f"of values changed")
  print(f"every hour paired with its T-{int(NWCSAF_SLOT_LAG.total_seconds()//60)}min "
        f"slot; {len(TIMES) - NT} of {len(TIMES)} model hours dropped for want of one")
if HIST_CACHE and _cached is None:
    print(f"histograms cached to {HIST_CACHE}")

## 7. Calibration and evaluation pools

Same split as the original -- daytime from 2025-10-07 on is the evaluation pool, everything else is
available for fitting -- with one change: the day/night cut is the satellite's own illumination flag rather
than a recomputed solar zenith angle, and **twilight is excluded from the evaluation pool entirely** rather
than being assigned to whichever side of a 90 degree cut it falls on. Twilight is where a cloud mask is
least reliable, and it is a small fraction of the sample, so it is not worth contaminating the headline
number with.

The hourly steps are not independent. The decorrelation time is measured from the autocorrelation of the
domain-mean bias and drives both the effective sample size and the block length of the bootstrap in
section 10.

In [ ]:
frac_day = np.array([H[("scan", "illum:day", "cloud")][i].sum()
                     + H[("scan", "illum:day", "clear")][i].sum() for i in range(NT)])
frac_twi = np.array([H[("scan", "illum:twilight", "cloud")][i].sum()
                     + H[("scan", "illum:twilight", "clear")][i].sum() for i in range(NT)])
tot = np.array([H[("scan", "all", "cloud")][i].sum()
                + H[("scan", "all", "clear")][i].sum() for i in range(NT)])

is_day = frac_day / np.maximum(tot, 1) > 0.5
is_twi = frac_twi / np.maximum(tot, 1) > 0.5
is_late = np.array([t >= SPLIT_DATE for t in TIMES_used])

# Day and night get their OWN threshold. tau* swings ~25 points over the
# diurnal cycle -- far more than the sampling uncertainty on any single value --
# so fitting one number on a night-heavy pool and applying it to daytime was the
# largest identifiable bias in the earlier design.
#
# Two classes only, and twilight counts as night: at solar zenith angles that
# low the visible channels are unusable, so CMA falls back to essentially its
# night-time IR scheme. Grouping it with day would mix two different retrievals.
#
# Both thresholds are calibrated on the days BEFORE the split and evaluated on
# the days after, so each is fitted and scored on the same illumination class.
is_night = ~is_day

CALIB_DAY = ~is_late & is_day
CALIB_NIGHT = ~is_late & is_night
EVAL_DAY = is_late & is_day
EVAL_NIGHT = is_late & is_night

POOLS = [("day", CALIB_DAY, EVAL_DAY), ("night", CALIB_NIGHT, EVAL_NIGHT)]

# pooled views, for the cells that report on one table (ladder, DWD, strata)
CALIB = CALIB_DAY | CALIB_NIGHT
EVAL = EVAL_DAY | EVAL_NIGHT

# --- decorrelation time from the domain-mean bias ---------------------------
def domain_mean_bias(cfg="scan+ct+plx"):
    hc, hk = H[(cfg, "all", "cloud")], H[(cfg, "all", "clear")]
    n = hc.sum(axis=1) + hk.sum(axis=1)
    mean_clct = (hc + hk) @ (TAUS / 100.0) / np.maximum(n, 1)
    obs = hc.sum(axis=1) / np.maximum(n, 1)
    return mean_clct - obs


bias_ts = domain_mean_bias()
bias_ts = bias_ts - np.mean(bias_ts)
n_lags = min(25, NT - 1)
acf = np.array([1.0 if k == 0 else np.corrcoef(bias_ts[:-k], bias_ts[k:])[0, 1]
                for k in range(n_lags)])
DECORR_H = int(next((k for k, a in enumerate(acf) if a < 1 / np.e), len(acf)))
DECORR_H = max(DECORR_H, 1)

fig, ax = plt.subplots(figsize=(6, 2.8))
ax.stem(range(len(acf)), acf)
ax.axhline(1 / np.e, color="k", ls="--", label="1/e")
ax.axvline(DECORR_H, color="r", ls=":", label=f"{DECORR_H} h")
ax.set_xlabel("lag (hours)")
ax.set_ylabel("ACF of domain-mean bias")
ax.legend()
plt.show()

# A pool can be empty if the window does not straddle SPLIT_DATE, or has no
# daytime hours. Say so here rather than letting np.nanargmax return NaN and
# fail three cells later with "cannot convert float NaN to integer".
for _nm, _c, _e in POOLS:
    if _c.sum() == 0 or _e.sum() == 0:
        raise RuntimeError(
            f"empty {_nm} pool: calibration {int(_c.sum())} h, evaluation "
            f"{int(_e.sum())} h. The window {TIMES_used[0]} .. {TIMES_used[-1]} "
            f"must straddle SPLIT_DATE ({SPLIT_DATE:%Y-%m-%d}) with {_nm} hours "
            f"on both sides.")

print(f"illumination from the product: {int(is_day.sum())} day, "
      f"{int(is_night.sum())} night (of which {int(is_twi.sum())} twilight)")
print(f"measured decorrelation time: {DECORR_H} h\n")
print(f"{'pool':<8s} {'calib h':>8s} {'eval h':>8s} {'eff N (eval)':>13s}")
for _nm, _c, _e in POOLS:
    eff = int(_e.sum()) / DECORR_H
    warn = f"  ** below {MIN_EFFECTIVE_N} independent samples **" if eff < MIN_EFFECTIVE_N else ""
    print(f"{_nm:<8s} {int(_c.sum()):8d} {int(_e.sum()):8d} {eff:13.0f}{warn}")

# time -> illumination class, so the frames can pick the matching threshold
ILLUM_BY_TIME = {t: ("day" if d else "night") for t, d in zip(TIMES_used, is_day)}

## 8. Youden's J and `tau*`

`J(tau) = POD(tau) - POFD(tau)`, maximised over the threshold. Peirce (1884) in its meteorological form,
Youden (1950) in general. Because it subtracts the false-alarm rate rather than dividing by a total, J is
insensitive to the base rate -- which matters here, where cloud covers something like 55% of pixels and a
score like the hit rate would be dominated by the climatology.

Evaluated straight from the histograms: `hits(tau)` is the reverse cumulative sum of the cloudy counts
from the bin upwards, `false alarms(tau)` the same over the clear counts. This is **exact**, not an
approximation of a direct contingency table -- bins are `floor(CLCT * 10)` and the threshold is
lower-inclusive, so `CLCT >= TAUS[k]` and `bin >= k` select identically the same pixels. The cell asserts
that identity against a brute-force count and raises if it ever drifts, because the entire notebook
downstream is built on it.

Reported alongside the point value: the **plateau**, the range of thresholds whose J is within one
bootstrap standard error of the maximum. A flat plateau means `tau*` is barely identified by the data, and
the point value should not be quoted to the nearest percent.

In [ ]:
def j_curve(hc, hk):
    """POD, POFD and J at every threshold, from summed histograms."""
    hc = np.asarray(hc, dtype="float64")
    hk = np.asarray(hk, dtype="float64")
    # lower-inclusive: pixels with bin >= k, i.e. CLCT >= TAUS[k]
    above_c = np.cumsum(hc[::-1])[::-1]
    above_k = np.cumsum(hk[::-1])[::-1]
    n_c, n_k = hc.sum(), hk.sum()
    pod = above_c / n_c if n_c > 0 else np.full(NBIN, np.nan)
    pofd = above_k / n_k if n_k > 0 else np.full(NBIN, np.nan)
    return pod, pofd, pod - pofd


def pooled(cfg, stratum, sel):
    return (H[(cfg, stratum, "cloud")][sel].sum(axis=0),
            H[(cfg, stratum, "clear")][sel].sum(axis=0))


def fit_tau(cfg, sel, stratum="all"):
    hc, hk = pooled(cfg, stratum, sel)
    _, _, J = j_curve(hc, hk)
    if not np.isfinite(J).any():
        return np.nan, np.nan, J
    k = int(np.nanargmax(J))
    return TAUS[k], float(J[k]), J


# --- the histogram identity, asserted rather than assumed -------------------
_rng = np.random.default_rng(0)
_c = _rng.uniform(0, 100, 200_000)
_o = (_rng.uniform(0, 1, 200_000) < (_c / 120 + 0.1)).astype(float)
_hc, _hk = hist_pair(_c, _o, np.ones(_c.size, bool))
_pod, _pofd, _ = j_curve(_hc, _hk)
_worst = 0.0
for _k in (0, NBIN // 7, NBIN // 2, NBIN - 1):   # scales with the grid
    _p, _ob = _c >= TAUS[_k], _o == 1
    _h, _m = (_p & _ob).sum(), (~_p & _ob).sum()
    _f, _n = (_p & ~_ob).sum(), (~_p & ~_ob).sum()
    _worst = max(_worst, abs(_h / (_h + _m) - _pod[_k]),
                 abs(_f / (_f + _n) - _pofd[_k]))
if _worst > 1e-12:
    raise RuntimeError(f"histogram J disagrees with a direct count by {_worst:.2e}")
print(f"histogram/contingency identity verified (max deviation {_worst:.1e})\n")

# The headline threshold: scan-time matched slot, snow/ice screened, parallax
# corrected, on the native CLCT that downstream products actually binarise.
BASE = "scan+ct+plx"

# --- one threshold per illumination class ----------------------------------
TAU, J_CURVE = {}, {}
for _nm, _c, _e in POOLS:
    tau_n, j_c, Jc = fit_tau(BASE, _c)
    _, _, Je = fit_tau(BASE, _e)
    k_n = int(np.rint(tau_n * BIN_PER_PCT))
    TAU[_nm], J_CURVE[_nm] = tau_n, (Jc, Je)
    pod, pofd, _ = j_curve(*pooled(BASE, "all", _e))
    print(f"{_nm:<6s} tau* = {tau_n:3.0f}%   J calib {j_c:.3f}   J eval {Je[k_n]:.3f}"
          f"   (POD {pod[k_n]:.3f}, POFD {pofd[k_n]:.3f})")

TAU_BY_ILLUM = dict(TAU)
tau_star, k_star = TAU["day"], int(np.rint(TAU["day"] * BIN_PER_PCT))
J_calib, J_eval = J_CURVE["day"]
j_calib = float(J_calib[k_star])      # day J at its own threshold
print(f"\nday - night = {TAU['day'] - TAU['night']:+.0f} points")
print("`tau_star` below is the DAY threshold; the ladder, DWD comparison and "
      "strata are\nreported on the daytime pool, which is the period of interest.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for _nm, col in zip(("day", "night"), ("C1", "C0")):
    Jc, _ = J_CURVE[_nm]
    axes[0].plot(TAUS, Jc, color=col, label=f"{_nm} (calibration)")
    axes[0].axvline(TAU[_nm], color=col, ls="--", label=f"{_nm} tau* = {TAU[_nm]:.0f}%")
axes[0].set_xlabel("CLCT threshold tau (%)")
axes[0].set_ylabel("J = POD - POFD")
axes[0].set_title(f"Threshold per illumination class, exp {EXP}")
axes[0].legend(fontsize=8)
for _nm, _c, _e in POOLS:
    pod, pofd, _ = j_curve(*pooled(BASE, "all", _e))
    k_n = int(np.rint(TAU[_nm] * BIN_PER_PCT))
    axes[1].plot(pofd, pod, label=f"{_nm} (evaluation)")
    axes[1].scatter([pofd[k_n]], [pod[k_n]], zorder=5, edgecolor="k")
axes[1].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[1].set_xlabel("POFD"); axes[1].set_ylabel("POD")
axes[1].set_title("ROC, evaluation pools")
axes[1].legend(fontsize=8)
plt.show()


## 9. The point of the notebook: how each correction moves `tau*`

The same fit repeated with the corrections switched on one at a time, cumulatively. Each row adds one
thing to the row above, so the change in `tau*` and in J is attributable to that one change. The last
column is the sample the row is fitted on -- screening and parallax both *remove* pixels, and a threshold
fitted on a smaller, cleaner sample is not automatically better, so the cost is shown next to the benefit.

What to read here: if `tau*` barely moves across the ladder, the original number was robust and the
corrections were not worth the loss of sample. If it moves by more than the bootstrap interval of section
11, the uncorrected threshold was absorbing a geometric or timing error, and any downstream use of it --
binarising CLCT for FSS or for SAL objects, as `CLCT_binary_verification.ipynb` does -- inherited that
error.

**The last row is different in kind and must not be read as a sixth point on the same axis.** The
detectability filter changes the forecast variable rather than the observation, so its optimum is a
threshold on filtered CLCT. In practice it collapses towards zero: once optically undetectable layers are
stripped, what remains is close to bimodal and "any cloud at all" becomes the best cut. That is a coherent
result, but the number is not on the same scale as the rows above and cannot be used to binarise the native
CLCT that FSS and SAL consume. Its **J** is the comparable quantity. The headline `tau*` therefore comes
from the last native-CLCT row, not from the bottom of the table.

In [ ]:
# The T-15 pairing is the baseline now, not a rung: it is the verified
# convention and is applied to every config. The old slot-T pairing is kept at
# the bottom as a labelled reference so the cost of the superseded convention
# stays visible, but it is no longer the starting point of the ladder.
# Fitted and scored on the DAYTIME pool only: the two illumination classes
# now have different thresholds, so a combined-pool ladder would mix them.
LADDER = [("scan", "T-15 slot, mask as delivered  [BASELINE]"),
          ("scan+ct", "+ snow/ice screening (ct classes 3, 4)"),
          ("scan+ct+plx", "+ parallax correction")]

rows = []
for cfg, label in LADDER:
    tau_c, j_c, _ = fit_tau(cfg, CALIB_DAY)
    if not np.isfinite(tau_c):
        rows.append((label, np.nan, np.nan, np.nan, 0))
        continue
    _, _, Je = fit_tau(cfg, EVAL_DAY)
    hc, hk = pooled(cfg, "all", EVAL_DAY)
    rows.append((label, tau_c, j_c, float(Je[int(np.rint(tau_c * BIN_PER_PCT))]),
                 int(hc.sum() + hk.sum())))

n0 = rows[0][4]
print(f"{'configuration':<52s} {'tau*':>7s} {'J calib':>8s} {'J eval':>8s} "
      f"{'eval pixels':>13s}")
print("-" * 92)
for label, tau_c, j_c, j_e, npx in rows:
    if not np.isfinite(tau_c):
        print(f"{label:<52s} {'not run':>7s}")
        continue
    frac = f"{100 * npx / n0:5.1f}%" if n0 else "   -- "
    print(f"{label:<52s} {tau_c:6.0f}% {j_c:8.3f} {j_e:8.3f} "
          f"{npx:9d} {frac}")

fig, ax = plt.subplots(figsize=(7, 4.2))
for cfg, label in LADDER:
    _, _, J = fit_tau(cfg, CALIB_DAY)
    ax.plot(TAUS, J, label=label.replace("+ ", ""), lw=1.4)
ax.set_xlabel("CLCT threshold tau (%)")
ax.set_ylabel("J on the calibration pool")
ax.set_title(f"Effect of each correction on the J curve, exp {EXP}")
ax.legend(fontsize=8)
plt.show()

## 10. How well is `tau*` actually determined? (moving-block bootstrap)

Resampling pixels would be meaningless -- neighbouring pixels are not independent draws and there are
hundreds of thousands of them, so any pixel-level interval would be absurdly narrow. The block bootstrap
resamples *time*, in contiguous blocks of the measured decorrelation length, which is the scale on which
this data actually carries new information. Because everything is precomputed histograms, a replicate is a
sum of `n/L` rows and a thousand of them cost nothing.

Two numbers come out, and the second is the more important:

- a confidence interval on `tau*` itself, and on `J(tau*)`;
- the **plateau**: every threshold whose calibration J lies within one bootstrap standard error of the
  maximum. If that plateau is 30 points wide, `tau*` is a convention, not a measurement, and quoting it to
  the nearest percent -- as the original notebook does -- reads far more precision into the data than is
  there.

In [ ]:
rng = np.random.default_rng(20251007)


def block_bootstrap(cfg, sel, n_boot=N_BOOT, L=None, stratum="all"):
    L = DECORR_H if L is None else L
    idx = np.where(sel)[0]
    n = idx.size
    if n < L:
        return np.array([]), np.array([])
    hc_all = H[(cfg, stratum, "cloud")][idx]
    hk_all = H[(cfg, stratum, "clear")][idx]
    n_blocks = max(int(np.ceil(n / L)), 1)
    starts_pool = np.arange(0, max(n - L + 1, 1))

    taus_b, js_b = np.empty(n_boot), np.empty(n_boot)
    for b in range(n_boot):
        st = rng.choice(starts_pool, size=n_blocks, replace=True)
        take = np.concatenate([np.arange(s, min(s + L, n)) for s in st])
        _, _, J = j_curve(hc_all[take].sum(axis=0), hk_all[take].sum(axis=0))
        if np.isfinite(J).any():
            k = int(np.nanargmax(J))
            taus_b[b], js_b[b] = TAUS[k], J[k]
        else:
            taus_b[b] = js_b[b] = np.nan
    return taus_b, js_b


taus_b, js_b = block_bootstrap(BASE, CALIB_DAY)
tau_lo, tau_hi = np.nanpercentile(taus_b, [2.5, 97.5])
j_lo, j_hi = np.nanpercentile(js_b, [2.5, 97.5])
se_j = float(np.nanstd(js_b))

plateau = TAUS[J_calib >= np.nanmax(J_calib) - se_j]
p_lo, p_hi = float(plateau.min()), float(plateau.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].hist(taus_b[np.isfinite(taus_b)], bins=40, color="C0")
axes[0].axvline(tau_star, color="k", ls="--")
axes[0].set_xlabel("bootstrap tau* (%)")
axes[0].set_ylabel("replicates")
axes[0].set_title(f"tau* = {tau_star:.0f}%, 95% CI [{tau_lo:.0f}, {tau_hi:.0f}]")

axes[1].plot(TAUS, J_calib, color="C0")
axes[1].axhspan(np.nanmax(J_calib) - se_j, np.nanmax(J_calib), color="C1", alpha=0.25)
axes[1].axvspan(p_lo, p_hi, color="C1", alpha=0.15)
axes[1].axvline(tau_star, color="k", ls="--")
axes[1].set_xlabel("CLCT threshold tau (%)")
axes[1].set_ylabel("J (calibration)")
axes[1].set_title(f"plateau within 1 SE: {p_lo:.0f} .. {p_hi:.0f}%")
plt.show()

print(f"block bootstrap on the DAY pool, {N_BOOT} replicates, block length "
      f"{DECORR_H} h ({int(CALIB_DAY.sum())} calibration hours)")
print(f"  tau*    = {tau_star:.0f}%   95% CI [{tau_lo:.0f}, {tau_hi:.0f}]  "
      f"(width {tau_hi - tau_lo:.0f} points)")
print(f"  J(tau*) = {j_calib:.3f}   95% CI [{j_lo:.3f}, {j_hi:.3f}]  SE {se_j:.4f}")
print(f"  plateau within 1 SE of the maximum: {p_lo:.0f} .. {p_hi:.0f}% "
      f"({p_hi - p_lo:.0f} points wide)")
if p_hi - p_lo > 20:
    print("  ** the J curve is flat: tau* is weakly identified and should be "
          "quoted as a range, not a value **")

## 11. Where a single threshold does not fit

One `tau*` for the whole domain and the whole period assumes the CLCT-to-cloud-mask relationship is the
same over the Plateau at 400 m and the Alps at 3000 m, and in daylight as at night. Refitting inside each
stratum tests that. It is a diagnostic, not a proposal to use a different threshold in each stratum --
that would fit a dozen parameters on a week of data. What it is for: if `tau*` differs sharply between
strata, the single global threshold is a compromise that is wrong nearly everywhere, and downstream
products built on it (binarised FSS, SAL objects) carry a spatially-structured error rather than a uniform
one. Elevation is the one to watch, because it is where the parallax displacement and the snow ambiguity
both bite hardest.

**Cloud class is handled separately, and not by refitting.** A stratum defined by the observed cloud type
*determines the observation*: `class:opaque` is about 99.5% cloudy, `class:snow_ice` almost entirely clear.
There is no meaningful mix of events and non-events inside such a stratum, so POD and POFD cannot both be
formed and a threshold fitted there would be circular by construction -- the earlier version of this
notebook printed "too few pixels to fit" for every class, which is that circularity showing up as a
symptom rather than being diagnosed.

The non-circular question is instead: **at the global `tau*`, how often does ICON put cloud where NWCSAF
sees each class of cloud?** That is a detection rate per class, fitted nowhere, and it is the more useful
number anyway -- it says which cloud types the model misses. The cloud-free classes get the complementary
quantity, the false-alarm rate. Both come from the unscreened configuration, since the `ct` screening
removes several of these classes from `BASE` by design.

In [ ]:
# --- (a) strata where a threshold can legitimately be fitted -----------------
FITTABLE = ["all"] + [f"elev:{n}" for n in elev_names] + \
           ["illum:day", "illum:night", "illum:twilight"]

print(f"{'stratum':<28s} {'tau* calib':>11s} {'J calib':>8s} {'J eval':>8s} "
      f"{'cloud frac':>11s} {'pixels':>12s}")
print("-" * 84)
strat_out = {}
for s_ in FITTABLE:
    hc_c, hk_c = pooled(BASE, s_, CALIB_DAY)
    if hc_c.sum() < 5000 or hk_c.sum() < 5000:
        print(f"{s_:<28s} {'too few pixels to fit':>11s}")
        continue
    _, _, Jc = j_curve(hc_c, hk_c)
    k_s = int(np.nanargmax(Jc))
    hc_e, hk_e = pooled(BASE, s_, EVAL_DAY)
    _, _, Je = j_curve(hc_e, hk_e)
    j_e_s = float(Je[k_s]) if np.isfinite(Je).any() else np.nan
    cf = hc_c.sum() / (hc_c.sum() + hk_c.sum())
    strat_out[s_] = (TAUS[k_s], float(Jc[k_s]), j_e_s)
    je_txt = f"{j_e_s:8.3f}" if np.isfinite(j_e_s) else f"{'--':>8s}"
    print(f"{s_:<28s} {TAUS[k_s]:10.0f}% {Jc[k_s]:8.3f} {je_txt} "
          f"{cf:11.3f} {int(hc_c.sum() + hk_c.sum()):12d}")

elev_taus = [strat_out[f"elev:{n}"][0] for n in elev_names if f"elev:{n}" in strat_out]
if len(elev_taus) > 1:
    spread = max(elev_taus) - min(elev_taus)
    print(f"\ntau* across elevation bands: {min(elev_taus):.0f} .. {max(elev_taus):.0f}% "
          f"(spread {spread:.0f} points; compare with the {tau_hi - tau_lo:.0f}-point "
          f"bootstrap CI on the global value)")
    if spread > tau_hi - tau_lo:
        print("  ** the elevation spread exceeds the sampling uncertainty on the "
              "global tau*: one\n     threshold for the whole domain is a compromise, "
              "and the error it leaves behind\n     is terrain-structured rather than "
              "uniform **")

# --- (b) cloud class: detection rate at the global tau*, not a refit ---------
# A class stratum determines the observation, so no threshold can be fitted in
# it. What is well defined is how often the model exceeds the global tau* on
# pixels of each observed class: a hit rate for the cloudy classes, a
# false-alarm rate for the cloud-free ones. Taken from the unscreened config,
# since the ct screening removes several of these classes from BASE.
print(f"\n\nDetection of each observed cloud class at the global tau* = "
      f"{tau_star:.0f}%  (config 'scan', unscreened)")
print(f"{'observed class':<28s} {'kind':>10s} {'n pixels':>11s} "
      f"{'CLCT >= tau*':>13s}")
print("-" * 66)
for s_ in [s for s in STRATA if s.startswith("class:")]:
    hc_c, hk_c = pooled("scan", s_, CALIB_DAY)
    n_c, n_k = hc_c.sum(), hk_c.sum()
    if n_c + n_k < 5000:
        print(f"{s_:<28s} {'--':>10s} {int(n_c + n_k):11d}   too few pixels")
        continue
    # the class decides which side it belongs on; use whichever is populated
    if n_c >= n_k:
        rate = j_curve(hc_c, hk_c)[0][k_star]      # POD
        kind, n_ = "cloudy", n_c
    else:
        rate = j_curve(hc_c, hk_c)[1][k_star]      # POFD
        kind, n_ = "clear", n_k
    print(f"{s_:<28s} {kind:>10s} {int(n_):11d} {rate:12.3f}")
print("\nFor the cloudy classes this is a hit rate (higher is better); for the "
      "clear ones a\nfalse-alarm rate (lower is better). A low hit rate on "
      "'thin' or 'fractional' is the\nsignature of cloud the model has but the "
      "satellite can barely see -- the same effect\nthe model-side filter in "
      "section 6 attacks from the other direction.")

# --- diurnal cycle ----------------------------------------------------------
# This project is about a diurnally varying spin-up bias, so the one question
# worth asking of the threshold is whether it moves with time of day. Free:
# the per-timestep histograms already exist, so this is a regrouping of what
# section 6 built, not a second pass over the data. All 144 hours are used
# (not the pools, which are day/night-skewed by construction), so it is a
# description of the period rather than an out-of-sample result.
hod = np.array([t.hour for t in TIMES_used])
TAU_BY_HOUR = {}          # hour -> tau* of its 3-hour bin, annotated on the frames
print(f"\n\nBy time of day, all {NT} hours, config '{BASE}'")
print(f"{'UTC':<10s} {'hours':>6s} {'tau*':>7s} {'J at own tau*':>14s} "
      f"{'J at its class':>14s} {'obs cloud':>10s}")
print("-" * 64)
for h0 in range(0, 24, 3):
    sel = np.isin(hod, range(h0, h0 + 3))
    if sel.sum() == 0:
        continue
    hc, hk = pooled(BASE, "all", sel)
    if hc.sum() < 5000 or hk.sum() < 5000:
        print(f"{h0:02d}-{h0+2:02d} UTC   {int(sel.sum()):6d}   too few pixels")
        continue
    _, _, J = j_curve(hc, hk)
    k_h = int(np.nanargmax(J))
    _cls = ("day" if sum(ILLUM_BY_TIME[t] == "day"
                         for t in np.array(TIMES_used)[sel]) * 2 >= sel.sum()
            else "night")
    for _h in range(h0, h0 + 3):
        TAU_BY_HOUR[_h] = float(TAUS[k_h])
    print(f"{h0:02d}-{h0+2:02d} UTC   {int(sel.sum()):6d} {TAUS[k_h]:6.0f}% "
          f"{J[k_h]:14.3f} "
          f"{J[int(np.rint(TAU_BY_ILLUM[_cls] * BIN_PER_PCT))]:14.3f} "
          f"{hc.sum()/(hc.sum()+hk.sum()):10.3f}")

_dt = [TAUS[int(np.nanargmax(j_curve(*pooled(BASE, "all", np.isin(hod, range(h, h + 3))))[2]))]
       for h in range(0, 24, 3)
       if pooled(BASE, "all", np.isin(hod, range(h, h + 3)))[0].sum() >= 5000]
if len(_dt) > 1:
    print(f"\ntau* across the diurnal cycle: {min(_dt):.0f} .. {max(_dt):.0f}% "
          f"(spread {max(_dt) - min(_dt):.0f} points; bootstrap CI on the global "
          f"value is {tau_hi - tau_lo:.0f} points)")


## 12. Fitted `tau*` against the DWD 12.5% threshold

A direct comparison of the threshold this notebook fits against **12.5%**, the value DWD reportedly uses.
That number is `1/8` -- one okta, the traditional synoptic unit of sky cover -- which is almost certainly
where it comes from, and it is a very different proposition from a fitted optimum: it declares a cell
cloudy as soon as an eighth of it is covered, where `tau*` lands somewhere above half.

Both are applied to the same ICON CLCT field, and both are shown twice:

- **all corrections** -- scan-matched slot, `ct` screening, parallax-displaced mask (the `BASE` config);
- **mask as delivered** -- the same verified `T-15` slot, but with no `ct` screening and no parallax
  displacement, so the two differ only in how the mask is processed.

A third row carries the **superseded slot-`T`** pairing (what `CLCT_binary_verification.ipynb`
compared against) so the cost of the old convention against the DWD threshold stays visible.

The corrections act on the observation, not on CLCT, so the model field is identical in both states. What
changes is the observation and its valid footprint, so the model panels are masked to each state's own
footprint -- otherwise the two rows would be scored over different areas and the comparison would be
meaningless.

Two things to expect, and they pull in opposite directions. At 12.5% the model declares cloud far more
often, so POD rises and POFD rises with it; whether that is a better trade is exactly what J measures.
And a low threshold interacts with the fringe effect: it pushes the model's cloud edge outward into the
partially-cloudy margins, so the disagreement should concentrate at cloud boundaries rather than
redistribute across the domain.

The period-wide scores come straight from the histograms already built in section 6, so they cost nothing.
Only the map panels reload fields, and only for the handful of timesteps listed in `N_SHOWCASE`.

In [ ]:
# =============================================================================
# Period-wide comparison, free of charge: both thresholds against both
# observation states, straight out of the section 6 histograms.
# =============================================================================
TAU_DWD = 12.5
k_dwd = int(round(TAU_DWD * BIN_PER_PCT))

if abs(TAUS[k_dwd] - TAU_DWD) > 1e-9:
    raise RuntimeError(f"threshold grid cannot represent {TAU_DWD}%")


def contingency_at(hc, hk, k):
    """Full contingency table and scores at one threshold bin."""
    hc = np.asarray(hc, dtype="float64")
    hk = np.asarray(hk, dtype="float64")
    n_c, n_k = hc.sum(), hk.sum()
    hits = np.cumsum(hc[::-1])[::-1][k]
    fa = np.cumsum(hk[::-1])[::-1][k]
    misses, cn = n_c - hits, n_k - fa
    pod = hits / n_c if n_c else np.nan
    pofd = fa / n_k if n_k else np.nan
    return dict(hits=hits, misses=misses, fa=fa, cn=cn, n=n_c + n_k,
                pod=pod, pofd=pofd, j=pod - pofd,
                far=fa / (hits + fa) if hits + fa else np.nan,
                csi=hits / (hits + misses + fa) if hits + misses + fa else np.nan,
                bias=(hits + fa) / n_c if n_c else np.nan,
                obs_frac=n_c / (n_c + n_k) if n_c + n_k else np.nan,
                mod_frac=(hits + fa) / (n_c + n_k) if n_c + n_k else np.nan)


# Both states use the verified T-15 slot, so they differ only in mask processing.
STATES = [("screen + parallax", BASE), ("mask as delivered", "scan")]
THRESH = [("fitted tau*", None), (f"DWD = {TAU_DWD}%", k_dwd)]

for pool_name, pool in (("daytime evaluation", EVAL_DAY),
                        ("night evaluation", EVAL_NIGHT)):
    print(f"\n{pool_name} ({int(pool.sum())} hours)")
    print(f"{'state':<16s} {'threshold':<16s} {'POD':>7s} {'POFD':>7s} {'J':>7s} "
          f"{'FAR':>7s} {'CSI':>7s} {'bias':>7s} {'obs cf':>8s} {'mod cf':>8s}")
    print("-" * 104)
    for sname, cfg in STATES:
        hc, hk = pooled(cfg, "all", pool)
        if hc.sum() + hk.sum() == 0:
            print(f"{sname:<16s} no data")
            continue
        for tname, kk in THRESH:
            _k = kk if kk is not None else int(np.rint(
                TAU["day" if "daytime" in pool_name else "night"] * BIN_PER_PCT))
            tname = (tname if kk is not None
                     else f"tau* = {TAUS[_k]:.0f}%")
            c = contingency_at(hc, hk, _k)
            print(f"{sname:<16s} {tname:<16s} {c['pod']:7.3f} {c['pofd']:7.3f} "
                  f"{c['j']:7.3f} {c['far']:7.3f} {c['csi']:7.3f} {c['bias']:7.3f} "
                  f"{c['obs_frac']:8.3f} {c['mod_frac']:8.3f}")

# --- where the two thresholds sit on the J curve ----------------------------
fig, ax = plt.subplots(figsize=(7, 4.2))
for (sname, cfg), col in zip(STATES, ["C0", "C3"]):
    hc, hk = pooled(cfg, "all", CALIB)
    _, _, J = j_curve(hc, hk)
    ax.plot(TAUS, J, color=col, label=f"{sname} (fitted on calibration)")
    ax.scatter([TAUS[k_star], TAU_DWD], [J[k_star], J[k_dwd]], color=col,
               zorder=5, s=45, edgecolor="k", linewidth=0.4)
ax.axvline(tau_star, color="k", ls="--", lw=1, label=f"tau* = {tau_star:.0f}%")
ax.axvline(TAU_DWD, color="k", ls=":", lw=1.2, label=f"DWD = {TAU_DWD}% (1 okta)")
ax.axvspan(p_lo, p_hi, color="C1", alpha=0.12, label="plateau (within 1 SE)")
ax.set_xlabel("CLCT threshold tau (%)")
ax.set_ylabel("J = POD - POFD")
ax.set_title(f"Where the DWD threshold sits on the J curve, exp {EXP}")
ax.legend(fontsize=8)
plt.show()

for sname, cfg in STATES:
    hc, hk = pooled(cfg, "all", CALIB)
    _, _, J = j_curve(hc, hk)
    inside = "inside" if p_lo <= TAU_DWD <= p_hi else "OUTSIDE"
    print(f"{sname:<16s}: J(tau*) = {J[k_star]:.3f}, J(12.5%) = {J[k_dwd]:.3f}, "
          f"deficit {J[k_star] - J[k_dwd]:+.3f}")
print(f"\n12.5% falls {inside} the {p_lo:.0f}-{p_hi:.0f}% plateau of the fitted "
      f"threshold.\nIf it is inside, the two thresholds are not distinguishable at "
      f"this sample size and the\nchoice between them does not matter for skill; if "
      f"outside, it is a real difference.")

### 12a. MOVERO first-guess CLCT bias, 801 and 802

The same verification series the other frame notebooks carry, read straight from the MOVERO ATAB
tables so that every frame below can show where in the period it sits. These are `FG-det` scores: each
row is a `+1 h` first guess verified against the surface network at its valid time, which is the same
clock the frames use.

The unit is worth pausing on. MOVERO reports CLCT bias in **octa**, and the DWD threshold of section 12
is `12.5% = 1/8` -- exactly one okta. So the two are commensurate: a first-guess bias of, say, `+0.5`
octa is half of the entire margin by which a one-okta threshold declares a cell cloudy. That is the
connection between the threshold question here and the bias this project is actually chasing.

Station subset `ch` (all Swiss stations); `VERIF_SUBSET` also accepts `ch-sp`, `ch-am`, `ch-av`.ipynb`; switch
`VERIF_SUBSET` to `ch`, `ch-am` or `ch-av` for all stations, the Alps or Alpine valleys.

In [ ]:
from pathlib import Path
import pandas as pd

# =============================================================================
# MOVERO verification series, for the bias panel under every frame.
# Same source and conventions as clc_section_frames.ipynb.
# =============================================================================
MOVERO_WD = Path("/scratch/mch/jdelbeke/movero/wd/2025_kenda_801_802")
EXP_IDS = ["801", "802"]
# Repo standard (Okabe-Ito, colourblind-safe), matching clc_section_frames,
# payerne_sounding_profiles, ramanlidar_profile_frames and
# CLCT_spindown_decomposition. Do not substitute tab:blue/tab:red -- frames
# from these notebooks get read side by side.
EXP_COLOUR = {"801": "#0072B2", "802": "#D55E00"}
VERIF_SUBSET = "ch"              # ch (all Swiss), ch-am (Alps), ch-sp (plateau), ch-av (valleys)
SUBSET_LABEL = {"ch": "all Swiss stations", "ch-am": "Alps", "ch-sp": "Swiss plateau",
                "ch-av": "Alpine valleys"}
VERIF_PARAM = "CLCT"
VERIF_SCORE = "ME"               # mean error = bias, in octa


def read_movero_time_scores(exp_dir, param=VERIF_PARAM):
    """One MOVERO ATAB time_scores file as a DataFrame indexed by valid time.

    The header block is variable length, so column names come from the row starting with
    YYYY rather than a fixed line number. The file's missing-value sentinel becomes NaN."""
    path = Path(exp_dir) / f"time_scores01_{param}.dat"
    with open(path) as fh:
        lines = fh.readlines()
    i_hdr = next(i for i, line in enumerate(lines) if line.split()[:1] == ["YYYY"])
    names = lines[i_hdr].split()
    df = pd.read_csv(path, sep=r"\s+", skiprows=i_hdr + 1, names=names, header=None)
    df = df.mask(df < -1e8)
    df["time"] = pd.to_datetime(dict(year=df["YYYY"], month=df["MM"], day=df["DD"],
                                     hour=df["hh"], minute=df["mm"]))
    return df.set_index("time").sort_index()


bias_df = {e: read_movero_time_scores(MOVERO_WD / f"{e}-FG-det_{VERIF_SUBSET}")
           for e in EXP_IDS}
bias = {e: df[VERIF_SCORE] for e, df in bias_df.items()}

for e, df in bias_df.items():
    print(f"{e}: {len(df)} rows, {df.index[0]:%d %b %H} - {df.index[-1]:%d %b %H} UTC, "
          f"{int(df['N'].max())} stations at most, {VERIF_SCORE} mean "
          f"{bias[e].mean():+.3f} octa ({bias[e].notna().sum()} valid hours)")

# Frozen across every frame, so the marker is comparable from one PNG to the next.
BIAS_TSPAN = (min(b.index[0] for b in bias.values()),
              max(b.index[-1] for b in bias.values()))
_all = np.concatenate([b.dropna().values for b in bias.values()])
_pad = 0.12 * (np.nanmax(_all) - np.nanmin(_all))
BIAS_YLIM = (np.nanmin(_all) - _pad, np.nanmax(_all) + _pad)


def draw_bias_panel(ax, t=None, legend=True):
    """The bias timeseries, optionally with the marker for time `t`. Axis limits are the
    frozen ones, not data-driven, so the marker sits identically in every frame."""
    for e in EXP_IDS:
        ax.plot(bias[e].index, bias[e].values, "-", color=EXP_COLOUR[e], lw=1.3,
                label=f"{e}  {VERIF_PARAM} {VERIF_SCORE}")
    ax.axhline(0, color="k", lw=0.9)
    ax.set_xlim(*BIAS_TSPAN)
    ax.set_ylim(*BIAS_YLIM)
    ax.grid(alpha=0.3)
    ax.set_ylabel(f"{VERIF_PARAM} bias [octa]")
    if legend:
        ax.legend(fontsize=8, ncol=len(EXP_IDS), loc="upper right")
    if t is None:
        return
    ts = pd.Timestamp(t)
    ax.axvline(ts, color="k", lw=1.4, ls=":", zorder=4)
    shown = []
    for e in EXP_IDS:
        val = bias[e].get(ts, np.nan)
        if np.isfinite(val):
            ax.plot([ts], [val], marker="*", ms=22, color=EXP_COLOUR[e],
                    mec="k", mew=0.9, zorder=6, clip_on=False)
            shown.append(f"{e} {val:+.2f}")
    ax.set_title(f"first-guess {VERIF_PARAM} bias, "
                 f"{SUBSET_LABEL.get(VERIF_SUBSET, VERIF_SUBSET)}; "
                 + ("now: " + ",  ".join(shown) + " octa" if shown
                    else "no verification at this hour"),
                 fontsize=10, loc="left")


FIGDIR = Path("./figures")
FIGDIR.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(16, 4))
draw_bias_panel(ax)
ax.set_title(f"First-guess {VERIF_PARAM} bias, "
             f"{SUBSET_LABEL.get(VERIF_SUBSET, VERIF_SUBSET)} ({VERIF_SUBSET}), "
             f"{VERIF_SCORE} in octa   [1 octa = the {TAU_DWD}% DWD threshold]",
             fontsize=11, loc="left")
ax.set_xlabel("valid time [UTC]")
fig.tight_layout()
fig.savefig(FIGDIR / "clct_threshold_movero_bias.png", dpi=120)
plt.show()
print(f"\nwrote {FIGDIR / 'clct_threshold_movero_bias.png'}")

### 12b. One frame per timestep, 801 against 802

Every hour in `TIMES_used` is drawn to its own PNG so the whole period can be stepped through rather
than sampled. Each frame is **one row per experiment**, four columns wide:

| column | what it shows |
|---|---|
| NWCSAF mask | the corrected observation: `T-15` slot, `ct`-screened, parallax-displaced |
| CLCT >= tau* | binarised at the fitted threshold |
| CLCT >= 12.5% | binarised at one okta |
| CLCT [%] | the forecast field itself, unthresholded -- the baseline both cuts are made from |

Continuous CLCT is placed **last** rather than beside the mask because it is the only panel that needs a
colour bar, and a colour bar in the middle of a row reads as a legend for the whole figure. On the right
edge it is unmistakably the scale for its own panel.

with the 801/802 MOVERO bias series underneath, marked at the current hour.

**Two changes from the earlier version, both deliberate.**

*The "mask as delivered" row is gone.* It differed from the corrected row mainly in which cells it
scored rather than in what the forecast did, so placing them side by side invited a comparison that was
really about sample composition -- section 9 shows the excluded cells are 1.4x as cloudy as the ones
kept. The corrected observation is the one retained because it is the more honest of the two: it marks
in amber exactly which cells the satellite could not resolve, instead of presenting a complete field and
leaving the reader to assume it is all equally trustworthy.

*Continuous CLCT is now shown.* Without it the two binarisations could only be read against each other,
which makes a threshold look like a property of the comparison rather than a cut through a continuous
field. With the raw field alongside, it is immediately visible how much of the domain sits between
12.5% and `tau*` -- which is exactly the population the choice of threshold decides the fate of.

**`tau*` is fitted on 801 and applied unchanged to 802.** Giving each experiment its own threshold would
make the two rows incomparable, which defeats the purpose; a single fitted cut applied to both is what
isolates the experiment difference. The per-frame J difference between the experiments is summarised at
the end of the cell, but the hours are not independent, so it is descriptive rather than a test.

Only the observation is drawn with gaps: ICON has no parallax and no missing cells, so the model panels
are complete over the whole domain. Scores are computed on the observed footprint, and each panel states
what fraction that was.

Step through them with the repo's viewer:

```bash
python3 scripts/frame_viewer.py figures/clct_threshold_frames
```

In [ ]:
import sys, subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed
from matplotlib.figure import Figure
from matplotlib.colors import ListedColormap

# =============================================================================
# One PNG per timestep: experiments 801 and 802 as rows, against the same
# corrected NWCSAF mask.
#
# The "mask as delivered" row was dropped. It differed from the corrected row
# mainly in which cells it scored rather than in the forecast, so side by side
# the two invited a comparison that was really about sample composition (see
# section 9 -- the excluded cells are 1.4x as cloudy as the ones kept). The
# corrected observation is kept because it is the honest one: it shows, in
# amber, exactly which cells the satellite could not resolve, instead of
# implying a complete field.
#
# Continuous CLCT is now shown alongside the two binarisations, so the
# thresholds can be read against the field they are cutting rather than only
# against each other.
#
# tau* is fitted on 801 (see section 8) and applied unchanged to 802 -- the
# point is to compare the two experiments under ONE threshold, not to give each
# its own.
# =============================================================================
FRAME_DIR = Path("./figures/clct_threshold_frames")
FRAME_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DPI = 110
N_WORKERS = 16
FRAME_EXPS = ["801", "802"]

# Four distinct claims, and conflating any two of them misleads:
#   clear / cloud            - a real observed or forecast state
#   not observed             - the satellite could not see a cell ICON covers
#                              (parallax gap or ct screening). OBSERVATION ONLY.
#   outside the ICON domain  - no model data at all. Left NaN -> white.
# The MODEL panels never carry "not observed": ICON has no parallax and no gaps.
CAT_CMAP = ListedColormap(["#e8e8e8", "#31465f", "#c98a3a"], name="cloudcat")
CAT_CMAP.set_bad("white")
NODATA_COLOUR = "#c98a3a"
CAT_CLEAR, CAT_CLOUD, CAT_UNOBS = 0, 1, 2

CLCT_CMAP = plt.get_cmap("Blues").copy()
CLCT_CMAP.set_bad("white")

_ii, _jj = np.where(in_domain)
_padx, _pady = 2 * abs(dlon), 2 * abs(dlat)
DOMAIN_EXTENT = [lon[_jj.min()] - _padx, lon[_jj.max()] + _padx,
                 lat[_ii.max()] - _pady, lat[_ii.min()] + _pady]


def icon_file_exp(exp, t):
    """lff path for an arbitrary experiment, derived from the configured one."""
    return f"{ICON_DIR.replace(f'/{EXP}/', f'/{exp}/')}/lff{t:%Y%m%d%H}"


def load_frame(t):
    """CLCT for every experiment, plus the corrected observation, for one hour."""
    clct = {}
    for e in FRAME_EXPS:
        fl_t = ekd.from_source("file", icon_file_exp(e, t)).to_fieldlist()
        xa_t = fl_t.sel({"parameter.variable": "CLCT"})[0].to_xarray()
        nm = "CLCT" if "CLCT" in xa_t else list(xa_t.data_vars)[0]
        clct[e] = remap(np.asarray(xa_t[nm].values, dtype="float64").ravel(),
                        sub, ti, ci, w, coverage, shape)

    obs = np.full(shape, np.nan)
    slot = match_slot(t)
    if slot is not None:
        Ss = read_nwcsaf(nwcsaf_file(slot), DOMAIN)
        screen = ~np.isin(Ss["ct"], CT_SNOW_ICE + CT_FRACTIONAL + CT_THIN)
        oc, _ = parallax_correct(Ss, HSURF_KM)
        obs = np.where(screen, oc, np.nan)
    return clct, obs, slot


def scores_2d(pred, obs, base):
    m = base & np.isfinite(obs)
    if not m.any():
        return np.nan, np.nan, np.nan
    p, o = pred[m], obs[m] == 1
    h, mi = (p & o).sum(), (~p & o).sum()
    f, c = (p & ~o).sum(), (~p & ~o).sum()
    pod = h / (h + mi) if h + mi else np.nan
    pofd = f / (f + c) if f + c else np.nan
    return pod, pofd, pod - pofd


def frame_path(t):
    return FRAME_DIR / f"clct_threshold_{t:%Y%m%d%H}.png"


def draw_frame(t):
    clct, obs, slot = load_frame(t)
    foot = in_domain & np.isfinite(obs)
    unobs = 1 - foot.sum() / max(int(in_domain.sum()), 1)
    out = {"time": t, "unobserved": unobs}

    # The binarisation uses the ONE global tau*, so frames stay comparable to
    # each other. tau* varies strongly with time of day (see the diurnal table),
    # so the hour's own optimum is annotated for contrast rather than applied.
    # Day and night use their own fitted threshold, so the cut changes with
    # illumination. The class is stated on the figure: an unexplained jump
    # between consecutive frames would otherwise look like a bug.
    _cls = ILLUM_BY_TIME.get(t, "day")
    _tau_used = TAU_BY_ILLUM[_cls]
    _tau_h = globals().get("TAU_BY_HOUR", {}).get(t.hour)

    nrow = len(FRAME_EXPS)
    # The maps have a FIXED aspect (cartopy + set_extent), so a hand-picked
    # figure height leaves a large dead band in every row: the domain is about
    # 2:1 wide, so a 4.4in panel is only ~2.2in of map however tall the row is.
    # Derive the row height from the domain instead of guessing it.
    FIG_W, NCOL = 19.5, 4
    _lon_span = DOMAIN_EXTENT[1] - DOMAIN_EXTENT[0]
    _lat_span = DOMAIN_EXTENT[3] - DOMAIN_EXTENT[2]
    _map_h = (FIG_W / NCOL) * 0.90 * _lat_span / _lon_span
    ROW_H = _map_h + 0.50          # + room for the two-line panel title
    LEG_H, BIAS_H, TOP_H = 0.50, 1.90, 0.40
    fig = Figure(figsize=(FIG_W, nrow * ROW_H + LEG_H + BIAS_H + TOP_H))
    gs = fig.add_gridspec(nrow + 2, NCOL,
                          height_ratios=[ROW_H] * nrow + [LEG_H, BIAS_H])
    row_axes = {}

    obs_cat = np.where(in_domain, np.where(foot, np.where(obs >= 0.5, CAT_CLOUD,
                                                          CAT_CLEAR), CAT_UNOBS), np.nan)
    for r, e in enumerate(FRAME_EXPS):
        c = clct[e]
        # Continuous CLCT sits last: it is the only panel carrying a colour
        # bar, and in the middle of the row that colour bar reads as a legend
        # for the whole figure. On the right edge it is unambiguously the
        # scale for its own panel.
        panels = [
            (obs_cat, "cat", f"NWCSAF mask (ct screening + parallax)\n"
                             f"slot {slot:%H:%M}   cloud fraction "
                             f"{np.nansum(obs_cat == CAT_CLOUD) / max(int(foot.sum()), 1):.3f}"
                             f"   unobserved {unobs:.1%}"),
            (c >= _tau_used, "tau",
             f"CLCT >= {_cls} tau* = {_tau_used:.0f}%"
             + (f"   (this hour's own optimum: {_tau_h:.0f}%)"
                if _tau_h is not None else "")),
            (c >= TAU_DWD, "dwd", f"CLCT >= {TAU_DWD}% (DWD, 1 okta)"),
            (c, "cont", f"CLCT [%] as forecast   domain mean "
                        f"{np.nanmean(np.where(in_domain, c, np.nan)):.1f}%"),
        ]
        for cix, (f_, kind, title) in enumerate(panels):
            ax = fig.add_subplot(gs[r, cix], projection=ccrs.PlateCarree())
            if kind == "cat":
                ax.pcolormesh(lon, lat, f_, transform=ccrs.PlateCarree(),
                              cmap=CAT_CMAP, vmin=-0.5, vmax=2.5, shading="nearest")
                sub_t = title
            elif kind == "cont":
                # Deliberately NOT fig.colorbar(..., ax=ax): on a fixed-aspect
                # cartopy axes that steals width from the map and sizes the bar
                # to the grid cell rather than to the map actually drawn, so the
                # bar comes out taller than its panel and the panel comes out
                # shorter than its neighbours. The scale goes in the legend
                # strip below instead, under the column it describes.
                cont_mesh = ax.pcolormesh(lon, lat, np.where(in_domain, f_, np.nan),
                                          transform=ccrs.PlateCarree(), cmap=CLCT_CMAP,
                                          vmin=0, vmax=100, shading="nearest")
                sub_t = title
            else:
                shown = np.where(in_domain,
                                 np.where(f_, CAT_CLOUD, CAT_CLEAR), np.nan)
                ax.pcolormesh(lon, lat, shown, transform=ccrs.PlateCarree(),
                              cmap=CAT_CMAP, vmin=-0.5, vmax=2.5, shading="nearest")
                pod, pofd, jj = scores_2d(np.asarray(f_), obs, foot)
                mcf = np.nansum(shown == CAT_CLOUD) / max(int(in_domain.sum()), 1)
                sub_t = (f"{title}   model cloud fraction {mcf:.3f}\n"
                         f"POD {pod:.3f}  POFD {pofd:.3f}  J {jj:.3f}"
                         f"   (scored on the {1 - unobs:.0%} observed)")
                out[f"{e}_{kind}_J"] = jj
            ax.coastlines("10m", linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.4)
            ax.set_extent(DOMAIN_EXTENT, crs=ccrs.PlateCarree())
            ax.set_title(sub_t, fontsize=8.5)
            if cix == 0:
                row_axes[r] = (ax, e)
        out[f"{e}_meanclct"] = float(np.nanmean(np.where(in_domain, c, np.nan)))

    from matplotlib.patches import Patch
    ax_leg = fig.add_subplot(gs[nrow, 0:3])
    ax_leg.axis("off")
    ax_leg.legend(handles=[Patch(fc="#e8e8e8", ec="k", lw=0.4, label="clear"),
                           Patch(fc="#31465f", ec="k", lw=0.4, label="cloud"),
                           Patch(fc=NODATA_COLOUR, ec="k", lw=0.4,
                                 label="not observed by the satellite "
                                       "(parallax gap / ct screening)"),
                           Patch(fc="white", ec="k", lw=0.4,
                                 label="outside the ICON domain")],
                  loc="center", ncol=4, fontsize=9, frameon=False,
                  title=f"same observation in both rows; day/night tau* fitted on "
                        f"{EXP} before {SPLIT_DATE:%d %b}, applied to both experiments",
                  title_fontsize=9.5)

    # An axes added straight into the cell fills it, which makes the bar as
    # tall as the whole legend strip. Inset a thin one inside a hidden host.
    cb_host = fig.add_subplot(gs[nrow, 3])
    cb_host.axis("off")
    cax = cb_host.inset_axes([0.12, 0.46, 0.76, 0.15])
    cb = fig.colorbar(cont_mesh, cax=cax, orientation="horizontal")
    cb.set_label("CLCT [%]", fontsize=8.5, labelpad=2)
    cax.tick_params(labelsize=7.5, pad=1)

    ax_bias = fig.add_subplot(gs[nrow + 1, :])
    draw_bias_panel(ax_bias, t)
    ax_bias.set_xlabel("valid time [UTC]")

    fig.suptitle(f"ICON CLCT vs NWCSAF cloud mask, {' vs '.join(FRAME_EXPS)}, "
                 f"{t:%Y-%m-%d %H:%M} UTC   [{_cls}]", fontsize=13)
    fig.tight_layout(rect=[0.014, 0, 1, 0.975], h_pad=0.4)
    # placed after tight_layout, from the axes' true positions, so the labels
    # track the rows whatever the computed geometry turns out to be
    for _r, (_ax0, _e) in row_axes.items():
        _pos = _ax0.get_position()
        fig.text(0.006, 0.5 * (_pos.y0 + _pos.y1), f"exp {_e}", rotation=90,
                 va="center", ha="center", fontsize=12, fontweight="bold")
    fig.savefig(frame_path(t), dpi=FRAME_DPI)
    return out


def _frame_job(t):
    try:
        return t, draw_frame(t), None
    except Exception as e:
        return t, None, f"{type(e).__name__}: {e}"


print(f"rendering {len(TIMES_used)} frames ({len(FRAME_EXPS)} experiments each) "
      f"on {N_WORKERS} workers -> {FRAME_DIR.resolve()}")
frame_rows, failed = [], []
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(_frame_job, t) for t in TIMES_used]
    for done, fut in enumerate(as_completed(futures), 1):
        t, res, err = fut.result()
        if err:
            failed.append((t, err))
            print(f"  FAILED {t:%m-%d %Hh}: {err}", flush=True)
        else:
            frame_rows.append(res)
        if done % 24 == 0:
            print(f"  {done}/{len(TIMES_used)}", flush=True)

print(f"\n{len(frame_rows)} frames written, {len(failed)} failed")

if frame_rows:
    fr = pd.DataFrame(frame_rows).sort_values("time").set_index("time")
    csv = FRAME_DIR / "frame_scores.csv"
    fr.to_csv(csv)
    print(f"per-frame scores -> {csv}")
    print(f"\nperiod means over {len(fr)} frames:")
    for c in [c for c in fr.columns if c.endswith("_J") or c.endswith("_meanclct")]:
        print(f"  {c:18s} {fr[c].mean():.3f}")
    if all(f"{e}_tau_J" in fr for e in FRAME_EXPS[:2]):
        a, b = FRAME_EXPS[0], FRAME_EXPS[1]
        d = fr[f"{a}_tau_J"] - fr[f"{b}_tau_J"]
        print(f"\n  J({a}) - J({b}) at tau*: mean {d.mean():+.4f}, "
              f"{a} better in {100 * (d > 0).mean():.0f}% of hours")
        print(f"  NOTE: the hours are not independent (~{DECORR_H} h decorrelation),")
        print(f"  so treat this as descriptive, not as a significance test.")

(FRAME_DIR / "provenance.txt").write_text(
    f"experiments {', '.join(FRAME_EXPS)} (rows)\n"
    f"period      {TIMES_used[0]} .. {TIMES_used[-1]} ({len(TIMES_used)} hours)\n"
    f"tau*        day {TAU['day']:.0f}% / night {TAU['night']:.0f}% -- fitted on "
    f"{EXP} before {SPLIT_DATE:%Y-%m-%d}\n"
    f"DWD tau     {TAU_DWD}% (1 okta)\n"
    f"observation NWCSAF cma, slot T-{int(NWCSAF_SLOT_LAG.total_seconds()//60)}min,\n"
    f"            ct screening + parallax (ctth_alti); amber = not observed\n"
    f"columns     mask | CLCT>=tau* | CLCT>=12.5% | CLCT [%]\n"
    f"bias panel  MOVERO {VERIF_PARAM} {VERIF_SCORE} [octa], {VERIF_SUBSET}, "
    f"801/802 FG-det\n"
    f"notebook    CLCT_threshold_corrected.ipynb\n")

fv = Path("frame_viewer.py")
if not fv.exists():
    fv = Path("scripts/frame_viewer.py")
if fv.exists():
    subprocess.run([sys.executable, str(fv), str(FRAME_DIR), "--write-only"], check=False)
    print(f"\nStep through them with:\n  python3 {fv} {FRAME_DIR}")
else:
    print("\nframe_viewer.py not found; frames are on disk and sort chronologically")

## 13. Summary

In [ ]:
print("=" * 88)
print(f"exp {EXP}   {TIMES_used[0]:%Y-%m-%d %H} .. {TIMES_used[-1]:%Y-%m-%d %H} UTC   "
      f"{NT} hours, grid {shape}, {int(in_domain.sum())} cells in domain")
print(f"config: '{BASE}'  (parallax={PARALLAX}, "
      f"slot lag={int(NWCSAF_SLOT_LAG.total_seconds()//60)} min, "
      f"screen={CT_SCREEN})")
print(f"pools: calibrated before {SPLIT_DATE:%Y-%m-%d}, evaluated after; "
      f"twilight counted as night")
print(f"effective sample size of the evaluation pool: ~{int(EVAL.sum()) / DECORR_H:.0f} "
      f"({DECORR_H} h decorrelation)")
print("-" * 88)
for _nm, _c, _e in POOLS:
    _k = int(np.rint(TAU[_nm] * BIN_PER_PCT))
    print(f"tau* {_nm:<6s}            {TAU[_nm]:3.0f}%   J eval "
          f"{J_CURVE[_nm][1][_k]:.3f}   ({int(_c.sum())} h calib, "
          f"{int(_e.sum())} h eval)")
print(f"day bootstrap CI        [{tau_lo:.0f}, {tau_hi:.0f}]   "
      f"plateau {p_lo:.0f}-{p_hi:.0f}%")
print(f"in-sample gap (day)     {np.nanmax(J_eval) - J_eval[k_star]:.4f}")
print("-" * 88)
print("movement of tau* along the correction ladder "
      "(last row thresholds a different variable):")
for label, tau_c, j_c, j_e, npx in rows:
    if np.isfinite(tau_c):
        print(f"  {label:<50s} {tau_c:6.0f}%  (J eval {j_e:.3f})")
    else:
        print(f"  {label:<50s} {'not run':>7s}")
print("=" * 88)
if tau_hi - tau_lo > 10:
    print("READ AS A RANGE, NOT A VALUE: the bootstrap interval and/or the fold "
          "spread are wide\nrelative to the movement along the ladder. Quote "
          "tau* to the nearest 5% at best.")

## Notes

**What is fitted where.** `tau*` is fitted on the calibration pool only, and every number reported on the
evaluation pool comes from that fit. The elevation band edges and the bootstrap block length are fixed a
priori or measured from the data rather than tuned against J;
`DECORR_H` is measured from the autocorrelation in section 7 and then used, not chosen.

**Corrections, and what each is really doing.**

- *Parallax* is applied to the observation, using the `ctth_alti` retrieval in the same file. It is a
  genuine correction for cloudy pixels with a height, an approximation for clear pixels (terrain height
  stands in for the surface), and nothing at all for the ~9% of cloudy pixels with no reliable retrieval,
  which are dropped. The gaps it opens behind displaced cloud are left as missing rather than filled.
- *Quality screening* could not be done the intended way: these exports carry no `cma_quality` or
  `cma_conditions`, so the cloud mask itself arrives with no confidence information. The `ctth_*` flags
  gate the height retrieval and hence the parallax correction; screening of the mask goes through the `ct`
  cloud-type classes instead. This is a weaker instrument than a real CMA confidence field and it is worth
  saying so. If the mask's own flags matter, they would have to be re-extracted from NWC SAF with the CMA
  ancillary variables retained.
- *Scan timing* removes a systematic ~10 minute offset, not a random one. It is the cheapest correction
  here and the only one that costs no sample.
- *Parallax* is not gated on `ctth_quality` by default. 28% of pixels carry a `bad` height flag, and
  strict gating leaves about a third of the domain unobserved; the ladder carries both so the choice is
  visible. Neither variant corrects the ~9% of cloudy pixels with no height retrieval at all.

**Known limitations.**
- The evaluation pool is a few days of daytime hours; the effective sample size is small enough that the
  differences along the correction ladder should be compared against the bootstrap interval before being
  believed, and 801-vs-802 differences almost certainly cannot be resolved at this sample size.
- The threshold grid is 0.1% and the histogram evaluation is exact rather than approximate, so threshold
  resolution is nowhere near the binding constraint -- the plateau width is.
- The `ct` screening removes fractional and thin-cirrus pixels from the fit. Those are real cloud, so the
  screened `tau*` answers "what threshold best separates cloud MSG can see confidently?" -- a slightly
  different question from the unscreened one. The ladder in section 9 is what makes that visible.
- One `tau*` is fitted for the whole period. Section 13 shows whether that is defensible; it does not fix
  it.

**Using `tau*` downstream.** `CLCT_binary_verification.ipynb` uses its `tau*` to binarise CLCT for the FSS
of section 6 and for the SAL objects of section 7. If the ladder here moves the threshold appreciably, both
of those should be re-run with the corrected value -- and given the plateau width, ideally re-run at the
plateau edges too, so that the sensitivity of the FSS and SAL conclusions to the threshold is visible
rather than assumed away.